# PC-Free-T4
Selecciona **Entorno de ejecución → GPU T4** y usa **Ejecutar todo** en una VM nueva.
Perfil gaming: **T4 + XFCE 720p + Selkies/NVENC H.264 + Turbo + 60 FPS objetivo + CBR 5 Mbps**.
**Flatpak/Sober activados**: preparación en segundo plano tras publicar el acceso,
condicionada al sandbox real. En la VM de desarrollo Colab bloqueó el montaje de `/proc`.
Firefox con EGL NVIDIA/WebRender se descarga en segundo plano; terminal, archivos, descargas e instalador de paquetes
Ubuntu están preparados. **No se abre ninguna aplicación automáticamente.**

Colab administrado restringe escritorio/control remoto: utiliza esto solo donde el proveedor
lo permita. No evita restricciones ni mantiene sesiones vivas, no monta Drive ni activa pagos.
La GPU, continuidad, compatibilidad de juegos y anticheat no están garantizados.
Las bibliotecas NVIDIA deben coincidir exactamente con el kernel y existir en los archivos
oficiales Ubuntu o el repositorio CUDA NVIDIA para Ubuntu 24.04 amd64; si no, se detiene
sin cambiar el controlador. No reejecutar sobre otro escritorio.

El enlace y la contraseña son privados, temporales y nuevos por sesión. **No guardar ni
compartir salidas del cuaderno**. El túnel expone una pantalla autenticada en Internet;
cerrar la VM al terminar. Pulsar **Jugar** en la web para solicitar captura del ratón y teclado.
FPS y bitrate son ajustables; rendimiento y latencia no garantizados.


In [ ]:
SOURCES = {'selkies-desktop.py': '"""Standalone Ubuntu 24.04/T4 desktop. Importing this module performs no setup.\n\nOnly bootstrap() installs/starts services. No notebook globals, Drive, keepalive,\ndriver installation, or automatic app windows. Logs stay on the ephemeral VM.\n"""\nimport base64\nimport contextlib\nimport fcntl\nimport json\nimport os\nimport platform\nimport pwd\nimport re\nimport secrets\nimport shlex\nimport shutil\nimport subprocess\nimport tarfile\nimport threading\nimport time\nimport urllib.error\nimport urllib.parse\nimport urllib.request\nfrom pathlib import Path\n\nBASE = Path(\'/tmp/pcfree-selkies\')\nUSER = \'pcfree-browser\'\nDISPLAY = \':98\'\nPORT = 8082\nSELKIES_VERSION = \'2.0.0\'\nSELKIES_DEB_VERSION = \'2.0.0-1~ubuntu24.04\'\nSELKIES_URL = (\'https://github.com/selkies-project/selkies/releases/download/\'\n               f\'{SELKIES_VERSION}/selkies-{SELKIES_VERSION}-ubuntu24.04-amd64.deb\')\nCLOUDFLARED_URL = \'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64\'\nFIREFOX_URL = \'https://download.mozilla.org/?product=firefox-latest-ssl&os=linux64&lang=es-ES\'\nPACKAGES = [\'xserver-xorg-core\', \'xfce4-session\', \'xfwm4\', \'xfce4-panel\',\n            \'xfdesktop4\', \'xfce4-settings\', \'thunar\', \'xfce4-terminal\',\n            \'dbus-x11\', \'mesa-utils\', \'vulkan-tools\', \'x11-utils\', \'x11-xserver-utils\', \'xauth\',\n            \'pulseaudio\', \'pulseaudio-utils\', \'ca-certificates\', \'python3.12\',\n            \'libgtk-3-0t64\', \'libdbus-glib-1-2\', \'libasound2t64\', \'zenity\', \'sudo\']\n\n\ndef private_json(path, value):\n    """Atomic, root-private state; no secrets in session.json."""\n    temporary = path.with_suffix(\'.tmp\')\n    fd = os.open(temporary, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)\n    with os.fdopen(fd, \'w\') as stream:\n        json.dump(value, stream, indent=2)\n    temporary.chmod(0o600)\n    temporary.replace(path)\n\n\ndef capture(argv, env=None, timeout=60):\n    return subprocess.check_output(argv, env=env, text=True, stderr=subprocess.STDOUT,\n                                   timeout=timeout)\n\n\ndef logged(argv, name=\'install\', env=None, timeout=600):\n    path = BASE / (name + \'.log\')\n    with path.open(\'a\') as stream:\n        result = subprocess.run(argv, env=env, stdout=stream, stderr=subprocess.STDOUT,\n                                timeout=timeout)\n    if result.returncode:\n        raise RuntimeError(f\'Falló {argv[0]}; consultar {path}.\\n\' + path.read_text()[-1800:])\n\n\ndef download(url, destination):\n    """HTTPS official publisher URLs; publish only a completed download."""\n    if urllib.parse.urlparse(url).scheme != \'https\':\n        raise ValueError(\'Se requiere HTTPS.\')\n    temporary = destination.with_suffix(destination.suffix + \'.part\')\n    try:\n        with urllib.request.urlopen(url, timeout=90) as response, temporary.open(\'wb\') as out:\n            if urllib.parse.urlparse(response.url).scheme != \'https\':\n                raise RuntimeError(\'Redirección insegura.\')\n            shutil.copyfileobj(response, out)\n        temporary.replace(destination)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\n@contextlib.contextmanager\ndef phase(name):\n    start = time.monotonic()\n    print(name + \'…\', flush=True)\n    yield\n    print(f\'  {time.monotonic() - start:.1f} s\', flush=True)\n\n\ndef wait_for(test, description, processes=(), timeout=45):\n    deadline = time.monotonic() + timeout\n    while time.monotonic() < deadline:\n        if any(process.poll() is not None for process in processes):\n            raise RuntimeError(f\'{description}: proceso detenido; registros en {BASE}.\')\n        try:\n            if test():\n                return\n        except (OSError, subprocess.SubprocessError, urllib.error.URLError):\n            pass\n        time.sleep(0.25)  # Readiness polling, never a session keepalive.\n    raise RuntimeError(f\'Tiempo agotado: {description}; registros en {BASE}.\')\n\n\ndef pid_identity(pid):\n    try:\n        # comm can contain spaces: fields after the closing parenthesis start at #3.\n        return Path(f\'/proc/{pid}/stat\').read_text().rsplit(\')\', 1)[1].split()[19]\n    except (OSError, IndexError):\n        return None\n\n\ndef already_running(session):\n    entries = session.get(\'processes\', {})\n    return bool(entries) and all(pid_identity(item[\'pid\']) == item[\'start\']\n                                 and item[\'start\'] is not None for item in entries.values())\n\n\ndef preflight():\n    if os.geteuid() != 0:\n        raise RuntimeError(\'Requiere una VM Ubuntu administrada con permisos root.\')\n    release = dict(line.split(\'=\', 1) for line in Path(\'/etc/os-release\').read_text().splitlines()\n                   if \'=\' in line)\n    if release.get(\'ID\', \'\').strip(\'"\') != \'ubuntu\' or release.get(\'VERSION_ID\', \'\').strip(\'"\') != \'24.04\':\n        raise RuntimeError(\'Esta receta requiere Ubuntu 24.04; no forzar otros paquetes.\')\n    if platform.machine() != \'x86_64\':\n        raise RuntimeError(\'Se requiere amd64.\')\n    if os.path.ismount(\'/content/drive\'):\n        raise RuntimeError(\'Desmontar Drive antes de preparar el escritorio.\')\n    if shutil.disk_usage(\'/tmp\').free < 8 * 1024**3:\n        raise RuntimeError(\'Se requieren al menos 8 GiB libres; los juegos necesitan más.\')\n    gpu = capture([\'nvidia-smi\', \'--query-gpu=name,driver_version,pci.bus_id\', \'--format=csv,noheader\']).strip().splitlines()\n    if len(gpu) != 1:\n        raise RuntimeError(\'Esta receta requiere una sola GPU Tesla T4.\')\n    name, driver, bus = (value.strip() for value in gpu[0].split(\',\'))\n    if \'T4\' not in name or not re.fullmatch(r\'\\d+\\.\\d+(?:\\.\\d+)?\', driver):\n        raise RuntimeError(\'GPU/versión NVIDIA no compatible.\')\n    if Path(\'/tmp/.X11-unix/X98\').exists() or Path(\'/tmp/.X98-lock\').exists():\n        raise RuntimeError(\'Ya existe Xorg :98. No se reinicia ni se reinstala la sesión activa.\')\n    print(f\'GPU: {name} | controlador del kernel: {driver}\', flush=True)\n    return driver, bus\n\n\ndef safe_apt_install(packages, validated_local_deb=None):\n    """Do not permit apt dependency resolution to replace a GPU/kernel stack."""\n    local_path = None\n    local_filename = None\n    if validated_local_deb is not None:\n        # Only the exact official download explicitly passed by bootstrap. Never\n        # permit arbitrary file: origins or infer trust from a package argument.\n        local_path = Path(validated_local_deb)\n        if (local_path != BASE / \'selkies.deb\' or str(local_path) not in packages\n                or local_path.is_symlink() or not local_path.is_file()):\n            raise RuntimeError(\'DEB local no validado; operación cancelada.\')\n        metadata = dict(line.split(\': \', 1) for line in\n                        capture([\'dpkg-deb\', \'-f\', str(local_path)]).splitlines() if \': \' in line)\n        if (metadata.get(\'Package\') != \'selkies\' or metadata.get(\'Version\') != SELKIES_DEB_VERSION\n                or metadata.get(\'Architecture\') != \'amd64\'):\n            raise RuntimeError(\'Identidad del DEB Selkies inesperada; operación cancelada.\')\n        local_filename = f"selkies_{SELKIES_DEB_VERSION}_amd64.deb"\n    plan = capture([\'apt-get\', \'-s\', \'install\', \'-y\', \'--no-install-recommends\', *packages], timeout=120)\n    for line in plan.splitlines():\n        if line.startswith(\'Remv \'):\n            raise RuntimeError(\'APT propone eliminar paquetes; operación cancelada.\')\n        if line.startswith(\'Inst \'):\n            name = line.split()[1].split(\':\')[0]\n            if \'nvidia\' in name or name.startswith((\'linux-image\', \'linux-modules\', \'linux-headers\')):\n                raise RuntimeError(f\'APT propone cambiar {name}; no se modifica el controlador del host.\')\n    origins = capture([\'apt-get\', \'--print-uris\', \'--download-only\', \'install\', \'-y\',\n                       \'--no-install-recommends\', *packages], timeout=120)\n    for uri, filename in re.findall(r"^\'([^\']*)\'\\s+(\\S+)", origins, re.MULTILINE):\n        parsed = urllib.parse.urlparse(uri)\n        if local_path is not None:\n            if (parsed.scheme == \'file\' and not parsed.netloc and not parsed.query\n                    and not parsed.fragment and urllib.parse.unquote(parsed.path) == str(local_path)):\n                continue\n            # Some APT versions omit the URI for an explicitly supplied local\n            # archive. Match its metadata-derived cache filename, not any \'\'.\n            if uri == \'\' and filename == local_filename:\n                continue\n        if not official_ubuntu_uri(uri):\n            raise RuntimeError(\'APT propone un origen no permitido; operación cancelada.\')\n    logged([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', *packages],\n           env={**os.environ, \'DEBIAN_FRONTEND\': \'noninteractive\'})\n\n\ndef matching_version(madison, driver):\n    for line in madison.splitlines():\n        columns = [part.strip() for part in line.split(\'|\')]\n        if len(columns) >= 3:\n            version = columns[1]\n            upstream = version.split(\':\')[-1].split(\'-\')[0]\n            if upstream == driver:\n                return version\n    raise RuntimeError(f\'Los repositorios oficiales no ofrecen bibliotecas NVIDIA {driver} exactas. \'\n                       \'Detenerse: no instalar otro controlador ni sustituir el del kernel.\')\n\n\ndef official_ubuntu_uri(uri):\n    """Ubuntu archives or NVIDIA\'s official CUDA APT repo for this exact OS/arch.\n\n    APT must already have authenticated the repository metadata; never add keys,\n    PPAs or trusted=yes, and never use a different driver version as fallback.\n    """\n    parsed = urllib.parse.urlparse(uri)\n    if parsed.username or parsed.password or parsed.port not in (None, 80, 443):\n        return False\n    ubuntu = (parsed.scheme in (\'http\', \'https\') and\n              (parsed.hostname in (\'archive.ubuntu.com\', \'security.ubuntu.com\', \'ports.ubuntu.com\')\n               or (parsed.hostname or \'\').endswith(\'.archive.ubuntu.com\')))\n    path = urllib.parse.unquote(parsed.path)\n    nvidia = (parsed.scheme == \'https\' and parsed.port in (None, 443) and\n              parsed.hostname == \'developer.download.nvidia.com\' and\n              path.startswith(\'/compute/cuda/repos/ubuntu2404/x86_64/\') and\n              \'..\' not in path.split(\'/\'))\n    return ubuntu or nvidia\n\n\ndef graphics(driver):\n    branch = driver.split(\'.\')[0]\n    packages = [f\'{name}-{branch}\' for name in (\'libnvidia-gl\', \'libnvidia-gpucomp\',\n                \'libnvidia-cfg1\', \'libnvidia-encode\', \'libnvidia-compute\', \'xserver-xorg-video-nvidia\')]\n    destination = BASE / \'nvidia\'; destination.mkdir(exist_ok=True)\n    # Resolve ALL versions and origins before extraction. Never install these debs.\n    selected = []\n    for package in packages:\n        version = matching_version(capture([\'apt-cache\', \'madison\', package]), driver)\n        spec = package + \'=\' + version\n        origins = capture([\'apt-get\', \'--print-uris\', \'download\', spec])\n        uris = re.findall(r"^\'([^\']+)\'", origins, re.MULTILINE)\n        if not uris or not all(official_ubuntu_uri(uri) for uri in uris):\n            raise RuntimeError(f\'{package}: origen no permitido (Ubuntu/NVIDIA oficial); operación cancelada.\')\n        selected.append(spec)\n    for spec in selected:\n        with (BASE / \'nvidia-download.log\').open(\'a\') as log:\n            subprocess.run([\'apt-get\', \'download\', spec], cwd=destination,\n                           stdout=log, stderr=subprocess.STDOUT, check=True, timeout=180)\n    root = destination / \'root\'\n    for deb in destination.glob(\'*.deb\'):\n        # dpkg-deb extraction executes no maintainer scripts and installs no driver.\n        logged([\'dpkg-deb\', \'-x\', str(deb), str(root)], \'nvidia-extract\')\n    libraries = sorted({str(path.parent) for path in root.rglob(\'*.so*\') if path.is_file()})\n    if not libraries or not list(root.rglob(\'nvidia_drv.so\')):\n        raise RuntimeError(\'Paquetes NVIDIA incompletos; revisar nvidia-download.log.\')\n    actual = capture([\'nvidia-smi\', \'--query-gpu=driver_version\', \'--format=csv,noheader\']).strip()\n    if actual != driver:\n        raise RuntimeError(\'El controlador del kernel cambió durante la preparación.\')\n    return root, \':\'.join(libraries)\n\n\ndef prepare_user(root, libraries):\n    try:\n        account = pwd.getpwnam(USER)\n    except KeyError:\n        logged([\'useradd\', \'--create-home\', \'--home-dir\', str(BASE / \'home\'), \'--shell\', \'/bin/bash\', USER])\n        account = pwd.getpwnam(USER)\n    if account.pw_uid == 0:\n        raise RuntimeError(\'El usuario de escritorio no puede ser root.\')\n    import grp\n    groups = []\n    for name in (\'video\', \'render\'):\n        try:\n            grp.getgrnam(name)\n            groups.append(name)\n        except KeyError:\n            pass\n    if groups:\n        logged([\'usermod\', \'-aG\', \',\'.join(groups), USER])\n    home = Path(account.pw_dir)\n    runtime = BASE / \'runtime\'\n    for directory in (home, runtime, home / \'Desktop\', home / \'Downloads\'):\n        directory.mkdir(parents=True, exist_ok=True)\n        os.chown(directory, account.pw_uid, account.pw_gid)\n    runtime.chmod(0o700)\n    authority = BASE / \'.Xauthority\'\n    authority.touch(mode=0o600); authority.chmod(0o600)\n    logged([\'xauth\', \'-f\', str(authority), \'add\', DISPLAY, \'MIT-MAGIC-COOKIE-1\', secrets.token_hex(16)])\n    os.chown(authority, account.pw_uid, account.pw_gid)\n    pulse_config = home / \'.config/pulse\'\n    pulse_config.mkdir(parents=True, exist_ok=True)\n    cookie = pulse_config / \'cookie\'\n    if not cookie.exists():\n        with cookie.open(\'xb\') as stream:\n            stream.write(secrets.token_bytes(256))\n    cookie.chmod(0o600)\n    for item in (home / \'.config\', pulse_config, cookie):\n        os.chown(item, account.pw_uid, account.pw_gid)\n    env = {**os.environ, \'DISPLAY\': DISPLAY, \'XAUTHORITY\': str(authority),\n           \'HOME\': str(home), \'USER\': USER, \'LOGNAME\': USER, \'XDG_RUNTIME_DIR\': str(runtime),\n           \'PULSE_SERVER\': f\'unix:{runtime}/pulse/native\', \'PULSE_COOKIE\': str(cookie), \'LD_LIBRARY_PATH\': libraries,\n           \'__GLX_VENDOR_LIBRARY_NAME\': \'nvidia\', \'SELKIES_INTERPOSER\': \'/usr/$LIB/selkies_input_interposer.so\'}\n    for key in (\'LD_PRELOAD\', \'DBUS_SESSION_BUS_ADDRESS\', \'SESSION_MANAGER\',\n                \'SELKIES_BASIC_AUTH_USER\', \'SELKIES_BASIC_AUTH_PASSWORD\', \'PASSWORD\', \'PASSWD\'):\n        env.pop(key, None)\n    env[\'DBUS_SESSION_BUS_ADDRESS\'] = f\'unix:path={runtime}/bus\'\n    # Persist only an explicit environment allowlist; never inherited Colab tokens.\n    keep = (\'DISPLAY\', \'XAUTHORITY\', \'HOME\', \'USER\', \'LOGNAME\', \'XDG_RUNTIME_DIR\',\n            \'PULSE_SERVER\', \'PULSE_COOKIE\', \'LD_LIBRARY_PATH\', \'__GLX_VENDOR_LIBRARY_NAME\',\n            \'SELKIES_INTERPOSER\', \'DBUS_SESSION_BUS_ADDRESS\')\n    session = dict(user=USER, home=str(home), runtime=str(runtime), display=DISPLAY,\n                   xauthority=str(authority), gpu_root=str(root),\n                   env={key: env[key] for key in keep}, run_launcher=\'/usr/local/bin/pcfree-run\')\n    return session, env\n\n\ndef vulkan_icd(root):\n    """Rewrite the official isolated NVIDIA ICD to its absolute extracted library.\n\n    Do not register it in /usr/share, where it could affect the managed host.\n    Keep the official API version and other fields; no guessed Vulkan version.\n    """\n    candidates = list(root.rglob(\'nvidia_icd.json\'))\n    if len(candidates) != 1:\n        raise RuntimeError(\'ICD Vulkan NVIDIA oficial no encontrado o ambiguo.\')\n    document = json.loads(candidates[0].read_text())\n    library = root / \'usr/lib/x86_64-linux-gnu/libGLX_nvidia.so.0\'\n    if not library.is_file() or not document.get(\'ICD\', {}).get(\'api_version\'):\n        raise RuntimeError(\'ICD/biblioteca Vulkan NVIDIA incompletos.\')\n    document[\'ICD\'][\'library_path\'] = str(library)\n    destination = BASE / \'nvidia-vulkan64.json\'\n    destination.write_text(json.dumps(document, indent=2) + \'\\n\')\n    destination.chmod(0o644)  # Not secret; the unprivileged Vulkan loader reads it.\n    return str(destination)\n\n\ndef as_user(argv):\n    return [\'runuser\', \'-u\', USER, \'--\', *argv]\n\n\ndef spawn(argv, name, env, processes, user=True):\n    with (BASE / (name + \'.log\')).open(\'a\') as log:\n        process = subprocess.Popen(as_user(argv) if user else argv, env=env,\n                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n    processes[name] = process\n    return process\n\n\ndef launcher_source(session):\n    # $LIB is expanded by the ELF loader per-process architecture, not by a shell.\n    # This handles 32-bit children of 64-bit Steam/Wine too, unlike probing argv[0].\n    return \'\'\'#!/usr/bin/python3.12\nimport json, os, pwd, sys\nif len(sys.argv) < 2:\n    raise SystemExit(\'Uso: pcfree-run programa [argumentos]\')\nsettings = SETTINGS\naccount = pwd.getpwnam(settings[\'user\'])\nif os.geteuid() not in (0, account.pw_uid):\n    raise SystemExit(\'Usuario de escritorio requerido.\')\nenv = dict(os.environ)\nenv.update(settings[\'env\'])\nenv.pop(\'LD_PRELOAD\', None)\n# Ubuntu amd64 glibc $LIB: lib/x86_64-linux-gnu; i386: lib/i386-linux-gnu.\nif os.path.exists(\'/usr/lib/x86_64-linux-gnu/selkies_input_interposer.so\'):\n    env[\'LD_PRELOAD\'] = \'/usr/$LIB/selkies_input_interposer.so\'\n    fake = \'/tmp/pcfree-selkies/fake-udev/\'\n    if all(os.path.exists(fake+\'lib/\'+arch+\'/libudev.so.1\')\n           for arch in (\'x86_64-linux-gnu\', \'i386-linux-gnu\')):\n        env[\'LD_PRELOAD\'] = fake+\'$LIB/libudev.so.1:\'+env[\'LD_PRELOAD\']\n        env.pop(\'SDL_JOYSTICK_DISABLE_UDEV\', None)\n        env.pop(\'SDL_JOYSTICK_DEVICE\', None)\n    else:\n        env[\'SDL_JOYSTICK_DISABLE_UDEV\'] = \'1\'\n        env[\'SDL_JOYSTICK_DEVICE\'] = \':\'.join(\'/dev/input/event\'+str(i) for i in range(1000,1004))\nelse:\n    print(\'Aviso: interposer no encontrado; mandos no preparados.\', file=sys.stderr)\nif os.geteuid() == 0:\n    os.initgroups(account.pw_name, account.pw_gid)\n    os.setgid(account.pw_gid)\n    os.setuid(account.pw_uid)\nos.execvpe(sys.argv[1], sys.argv[1:], env)\n\'\'\'.replace(\'SETTINGS\', repr(session))\n\n\ndef shortcut(session, filename, name, command, icon, terminal=False):\n    path = Path(session[\'home\']) / \'Desktop\' / (filename + \'.desktop\')\n    path.write_text(\'[Desktop Entry]\\nVersion=1.0\\nType=Application\\n\'\n                    f\'Name={name}\\nExec={command}\\nIcon={icon}\\nTerminal={str(terminal).lower()}\\n\')\n    path.chmod(0o755)\n    account = pwd.getpwnam(session[\'user\']); os.chown(path, account.pw_uid, account.pw_gid)\n\n\ndef desktop_files(session):\n    path = Path(session[\'home\']) / \'.config/xfce4/xfconf/xfce-perchannel-xml\'\n    path.mkdir(parents=True, exist_ok=True)\n    (path / \'xfwm4.xml\').write_text(\'\'\'<?xml version="1.0" encoding="UTF-8"?>\n<channel name="xfwm4" version="1.0"><property name="general" type="empty">\n<property name="use_compositing" type="bool" value="false"/>\n</property></channel>\n\'\'\')\n    account = pwd.getpwnam(USER)\n    config_home = Path(session[\'home\']) / \'.config\'\n    for item in [config_home, *config_home.rglob(\'*\')]:\n        os.chown(item, account.pw_uid, account.pw_gid)\n    runner = Path(session[\'run_launcher\']); runner.write_text(launcher_source(session)); runner.chmod(0o755)\n    inputs = Path(\'/dev/input\')\n    if not inputs.exists():\n        inputs.mkdir(mode=0o1777)\n        inputs.chmod(0o1777)\n    shortcut(session, \'Terminal\', \'Terminal\', \'pcfree-run xfce4-terminal\', \'utilities-terminal\')\n    shortcut(session, \'Archivos\', \'Archivos\', \'pcfree-run thunar\', \'system-file-manager\')\n    shortcut(session, \'Descargas\', \'Descargas\', \'pcfree-run thunar \' + session[\'home\'] + \'/Downloads\', \'folder-download\')\n    package_installer(session)\n\n\ndef package_installer(session):\n    """Interactive, small allowlist: no unrestricted passwordless sudo/apt."""\n    allowed = [\'mousepad\', \'gimp\', \'vlc\', \'p7zip-full\', \'unzip\', \'file-roller\', \'htop\']\n    helper = Path(\'/usr/local/sbin/pcfree-install-package\')\n    # Root-owned script; official origins and safe dependency plan.\n    helper.write_text(\'\'\'#!/usr/bin/python3.12\nimport os, re, subprocess, sys, urllib.parse\ndef official_ubuntu_uri(uri):\n    parsed = urllib.parse.urlparse(uri)\n    if parsed.username or parsed.password or parsed.port not in (None, 80, 443):\n        return False\n    host = parsed.hostname or \'\'\n    ubuntu = parsed.scheme in (\'http\', \'https\') and (\n        host in (\'archive.ubuntu.com\',\'security.ubuntu.com\',\'ports.ubuntu.com\')\n        or host.endswith(\'.archive.ubuntu.com\'))\n    path = urllib.parse.unquote(parsed.path)\n    nvidia = (parsed.scheme == \'https\' and parsed.port in (None, 443)\n        and host == \'developer.download.nvidia.com\'\n        and path.startswith(\'/compute/cuda/repos/ubuntu2404/x86_64/\')\n        and \'..\' not in path.split(\'/\'))\n    return ubuntu or nvidia\nALLOWED = ALLOWLIST\nif os.geteuid() != 0 or len(sys.argv) != 2 or sys.argv[1] not in ALLOWED:\n    raise SystemExit(\'Seleccionar un paquete permitido desde el icono Instalador.\')\nplan = subprocess.check_output([\'apt-get\',\'-s\',\'install\',\'-y\',\'--no-install-recommends\',sys.argv[1]], text=True)\nfor line in plan.splitlines():\n    if line.startswith(\'Remv \') or (line.startswith(\'Inst \') and\n       (\'nvidia\' in line.split()[1] or line.split()[1].startswith((\'linux-image\',\'linux-modules\',\'linux-headers\')))):\n        raise SystemExit(\'Operación insegura: cambios del controlador/kernel o eliminaciones.\')\nuris = subprocess.check_output([\'apt-get\',\'--print-uris\',\'--download-only\',\'install\',\'-y\',\'--no-install-recommends\',sys.argv[1]], text=True)\nfor uri in re.findall(r"^\'([^\']+)\'", uris, re.MULTILINE):\n    if not official_ubuntu_uri(uri):\n        raise SystemExit(\'Origen no permitido: solamente Ubuntu o NVIDIA CUDA oficial Ubuntu24.04 amd64.\')\nsubprocess.run([\'apt-get\',\'install\',\'-y\',\'--no-install-recommends\',sys.argv[1]], check=True,\n               env={**os.environ,\'DEBIAN_FRONTEND\':\'noninteractive\'})\n\'\'\'.replace(\'ALLOWLIST\', repr(allowed)))\n    helper.chmod(0o755)\n    rule = Path(\'/etc/sudoers.d/pcfree-packages\')\n    rule.write_text(f\'{USER} ALL=(root) NOPASSWD: /usr/local/sbin/pcfree-install-package *\\n\')\n    rule.chmod(0o440)\n    logged([\'visudo\', \'-cf\', str(rule)])\n    gui = Path(\'/usr/local/bin/pcfree-install\')\n    gui.write_text(\'#!/bin/sh\\npackage=$(zenity --list --title="Paquetes Ubuntu" --column="Paquete" \'\n                   + \' \'.join(shlex.quote(item) for item in allowed) + \') || exit\\n\'\n                   + \'sudo -n /usr/local/sbin/pcfree-install-package "$package"\\nread -r _\\n\')\n    gui.chmod(0o755)\n    shortcut(session, \'Instalador\', \'Instalador de paquetes Ubuntu\', \'pcfree-run pcfree-install\', \'system-software-install\', True)\n\n\ndef firefox_gpu_profile(session):\n    """Dedicated profile; preserve accounts and settings of other profiles."""\n    profile = BASE / \'firefox-profile\'\n    profile.mkdir(mode=0o700, exist_ok=True)\n    preferences = \'\\n\'.join([\n        \'user_pref("gfx.webrender.all", true);\',\n        \'user_pref("layers.acceleration.force-enabled", true);\',\n        \'user_pref("webgl.disabled", false);\',\n        \'user_pref("webgl.force-enabled", true);\',\n        \'user_pref("browser.sessionstore.resume_from_crash", false);\',\n    ]) + \'\\n\'\n    (profile / \'user.js\').write_text(preferences)\n    account = pwd.getpwnam(session[\'user\'])\n    for item in (profile, profile / \'user.js\'):\n        os.chown(item, account.pw_uid, account.pw_gid)\n    return profile\n\n\ndef firefox_background(session):\n    def worker():\n        try:\n            destination = BASE / \'firefox\'; destination.mkdir(exist_ok=True)\n            archive = destination / \'firefox.tar\'\n            if not (destination / \'firefox/firefox\').exists():\n                download(FIREFOX_URL, archive)\n                with tarfile.open(archive) as bundle:\n                    bundle.extractall(destination, filter=\'data\')\n                archive.unlink()\n            command = \'/usr/local/bin/pcfree-firefox\'\n            profile = firefox_gpu_profile(session)\n            Path(command).write_text(\'#!/bin/sh\\nexec /usr/local/bin/pcfree-run \'\n                                     + \'env MOZ_X11_EGL=1 \'\n                                     + shlex.quote(str(destination / \'firefox/firefox\'))\n                                     + \' --no-remote --profile \' + shlex.quote(str(profile)) + \' "$@"\\n\')\n            Path(command).chmod(0o755)\n            shortcut(session, \'Firefox\', \'Firefox\', command + \' %u\',\n                     str(destination / \'firefox/browser/chrome/icons/default/default128.png\'))\n            (BASE / \'firefox-status.txt\').write_text(\'listo\\n\')\n            print(\'Firefox listo: icono instalado; no se abrió ninguna ventana.\', flush=True)\n        except Exception as error:\n            (BASE / \'firefox-status.txt\').write_text(\'error: \' + str(error) + \'\\n\')\n            print(f\'Firefox: descarga/instalación fallida: {error}. Reintentar con firefox_background(SESSION).\', flush=True)\n    thread = threading.Thread(target=worker, name=\'pcfree-firefox\', daemon=True)\n    thread.start()\n    return thread\n\n\ndef show_access():\n    access = json.loads((BASE / \'access.json\').read_text())\n    print(\'\\nAcceso privado — no guardar estas salidas en GitHub:\', flush=True)\n    print(\'URL:\', access[\'url\'], \'\\nUsuario:\', access[\'username\'], \'\\nContraseña:\', access[\'password\'], flush=True)\n    state = BASE / \'session.json\'\n    settings = json.loads(state.read_text()).get(\'settings\', {}) if state.exists() else {}\n    fps, kbps = settings.get(\'framerate\', 30), settings.get(\'bitrate\', 2000)\n    print(f\'720p · H.264 · {fps} fps · {kbps} kbps iniciales. Ajustar según red; latencia no garantizada.\')\n    firefox = \'descarga en segundo plano\' if settings.get(\'firefox\', True) else \'desactivado\'\n    print(f\'Registros: {BASE} | Firefox: {firefox}, sin abrir ventanas.\')\n\n\ndef auth_ready(username, password):\n    url = f\'http://127.0.0.1:{PORT}/\'\n    try:\n        urllib.request.urlopen(url, timeout=2).close()\n        raise RuntimeError(\'Selkies sirve contenido sin autenticación; no publicar túnel.\')\n    except urllib.error.HTTPError as error:\n        if error.code != 401:\n            return False\n    header = base64.b64encode(f\'{username}:{password}\'.encode()).decode()\n    request = urllib.request.Request(url, headers={\'Authorization\': \'Basic \' + header})\n    with urllib.request.urlopen(request, timeout=2) as response:\n        return response.status == 200\n\n\nINPUT_GUARD_JS = r\'\'\'(() => {\n  // Only cancel browser defaults over the stream, never stop input forwarding.\n  // OS-level Android BACK and reserved system shortcuts cannot be intercepted.\n  const streamTarget = target => target instanceof Element &&\n    !!target.closest(\'#overlayInput, #videoCanvas, #videoWorkerCanvas, #videoStream, #stream\');\n  document.addEventListener(\'mousedown\', event => {\n    if (event.button === 2 && streamTarget(event.target)) event.preventDefault();\n  }, {capture: true, passive: false});\n  document.addEventListener(\'contextmenu\', event => {\n    if (streamTarget(event.target)) event.preventDefault();\n  }, {capture: true, passive: false});\n  window.pcfreeInputGuard = {version: 1, osBackCapture: false};\n  const addGamingButton = () => {\n    const nativeButton = document.querySelector(\'button.gaming-mode-button\');\n    if (!nativeButton || document.getElementById(\'pcfree-play\')) return !!nativeButton;\n    const button = document.createElement(\'button\');\n    button.id = \'pcfree-play\'; button.type = \'button\'; button.textContent = \'Jugar\';\n    button.title = \'Pantalla completa y captura de controles; requiere permisos del navegador\';\n    button.style.cssText = \'position:fixed;bottom:12px;left:12px;z-index:10000;padding:8px 16px;background:#166534;color:white;border:1px solid #86efac;border-radius:8px;cursor:pointer\';\n    button.addEventListener(\'click\', () => {\n      // Use the upstream handler within this real user-activated click.\n      if (typeof window.webrtcInput?.enterGamingMode === \'function\') window.webrtcInput.enterGamingMode();\n      else document.querySelector(\'button.gaming-mode-button\')?.click();\n    });\n    const visibility = () => { button.hidden = !!document.fullscreenElement; };\n    document.addEventListener(\'fullscreenchange\', visibility);\n    document.body.appendChild(button); visibility();\n    return true;\n  };\n  const start = () => {\n    if (addGamingButton()) return;\n    // Only the dashboard needs the CTA. Input-only iframe stays lightweight.\n    if (window.self !== window.top) return;\n    const observer = new MutationObserver(() => { if (addGamingButton()) observer.disconnect(); });\n    observer.observe(document.body, {childList: true, subtree: true});\n  };\n  if (document.readyState === \'loading\') document.addEventListener(\'DOMContentLoaded\', start, {once: true});\n  else start();\n})();\n\'\'\'\n\n\ndef prepare_web_client(source=None):\n    """Private static copy; keep upstream bundles and native Gaming Mode intact."""\n    source = Path(source or \'/opt/selkies/lib/python3.12/site-packages/selkies/selkies_web\')\n    if not (source / \'index.html\').is_file() or not (source / \'assets\').is_dir():\n        raise RuntimeError(\'Cambió la estructura del cliente Selkies; revisar paquete antes de aplicar controles.\')\n    target = BASE / \'web-client\'\n    shutil.copytree(source, target, dirs_exist_ok=True)\n    marker = \'<script src="./pcfree-input-guard.js"></script>\'\n    for index in target.glob(\'*.html\'):\n        html = index.read_text()\n        if \'</head>\' not in html:\n            raise RuntimeError(\'Cliente web sin cabecera reconocible; no modificar bundles.\')\n        if marker not in html:\n            index.write_text(html.replace(\'</head>\', marker + \'\\n</head>\', 1))\n    (target / \'pcfree-input-guard.js\').write_text(INPUT_GUARD_JS)\n    return str(target)\n\n\ndef bootstrap(firefox=True, framerate=60, bitrate=5000):\n    if not isinstance(framerate, int) or not 1 <= framerate <= 60 or not isinstance(bitrate, int) or not 500 <= bitrate <= 20000:\n        raise ValueError(\'FPS: 1–60; bitrate: 500–20000 kbps.\')\n    BASE.mkdir(mode=0o755, exist_ok=True)\n    with (BASE / \'bootstrap.lock\').open(\'w\') as lock:\n        try:\n            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError:\n            raise RuntimeError(\'Otra preparación está en curso; no ejecutar dos veces.\')\n        state = BASE / \'session.json\'\n        if state.exists():\n            session = json.loads(state.read_text())\n            if already_running(session):\n                print(\'Sesión ya activa; no se reinicia ni se reinstala.\')\n                show_access()\n                return session\n            raise RuntimeError(\'Estado previo incompleto/detenido. Revisar registros; no reiniciar automáticamente. Usar VM nueva.\')\n        started = time.monotonic()\n        with phase(\'Comprobaciones\'):\n            driver, bus = preflight()\n        with phase(\'Paquetes mínimos y Selkies oficial\'):\n            logged([\'apt-get\', \'update\', \'-qq\'], timeout=240)\n            safe_apt_install(PACKAGES)\n            deb = BASE / \'selkies.deb\'; download(SELKIES_URL, deb)\n            safe_apt_install([str(deb)], validated_local_deb=deb)\n            # Only fix the private package venv, NEVER /usr/bin/python3.\n            python = Path(\'/opt/selkies/bin/python3\')\n            if not python.is_symlink():\n                raise RuntimeError(\'Python privado Selkies inesperado; no modificar Python del sistema.\')\n            python.unlink(); python.symlink_to(\'/usr/bin/python3.12\')\n            capture([\'/opt/selkies/bin/selkies\', \'--help\'])\n            web_root = prepare_web_client()\n            tunnel = BASE / \'cloudflared\'; download(CLOUDFLARED_URL, tunnel); tunnel.chmod(0o755)\n        with phase(\'Bibliotecas 3D exactas, aisladas del kernel\'):\n            root, libraries = graphics(driver)\n            session, env = prepare_user(root, libraries)\n            icd = vulkan_icd(root)\n            for key in (\'VK_DRIVER_FILES\', \'VK_ICD_FILENAMES\'):\n                env[key] = icd\n                session[\'env\'][key] = icd\n            egl = BASE / \'nvidia-egl.json\'\n            egl_library = next(root.rglob(\'libEGL_nvidia.so.0\'), None)\n            if egl_library is None:\n                raise RuntimeError(\'Falta EGL NVIDIA para el navegador; no usar renderizado software silencioso.\')\n            egl.write_text(json.dumps({\'file_format_version\': \'1.0.0\',\n                                      \'ICD\': {\'library_path\': str(egl_library)}}))\n            for mapping in (env, session[\'env\']):\n                mapping[\'__EGL_VENDOR_LIBRARY_FILENAMES\'] = str(egl)\n            session[\'driver_version\'] = driver\n            session[\'settings\'] = {\'framerate\': framerate, \'bitrate\': bitrate, \'firefox\': firefox}\n            desktop_files(session)\n        # Mark setup BEFORE launching, so failures never cause a destructive rerun.\n        session[\'processes\'] = {}\n        private_json(state, session)\n        processes = {}\n        with phase(\'Xorg NVIDIA, XFCE y audio\'):\n            parts = bus.split(\':\'); device, function = parts[-1].split(\'.\')\n            busid = f\'PCI:{int(parts[-2],16)}:{int(device,16)}:{int(function,16)}\'\n            config = BASE / \'xorg.conf\'\n            config.write_text(\'\'\'Section "ServerFlags"\n Option "AutoAddDevices" "false"\n Option "AutoAddGPU" "false"\nEndSection\nSection "Device"\n Identifier "T4"\n Driver "nvidia"\n BusID "BUSID"\n Option "AllowEmptyInitialConfiguration" "true"\nEndSection\nSection "Screen"\n Identifier "GPU-Screen"\n Device "T4"\n DefaultDepth 24\n SubSection "Display"\n  Depth 24\n  Virtual 1280 720\n EndSubSection\nEndSection\n\'\'\'.replace(\'BUSID\', busid))\n            module = next(root.rglob(\'nvidia_drv.so\')).parent.parent\n            xorg = spawn([\'/usr/lib/xorg/Xorg\', DISPLAY, \'-config\', str(config), \'-modulepath\',\n                          f\'{module},/usr/lib/xorg/modules\', \'-logfile\', str(BASE / \'Xorg.log\'),\n                          \'-nolisten\', \'tcp\', \'-noreset\', \'-novtswitch\', \'-sharevts\', \'-auth\',\n                          session[\'xauthority\']], \'xorg\', env, processes, user=False)\n            def display_ready():\n                return subprocess.run(as_user([\'xdpyinfo\']), env=env, stdout=subprocess.DEVNULL,\n                                      stderr=subprocess.DEVNULL, timeout=3).returncode == 0\n            wait_for(display_ready, \'Xorg\', [xorg])\n            renderer = capture(as_user([\'glxinfo\', \'-B\']), env=env)\n            (BASE / \'opengl.log\').write_text(renderer)\n            if \'NVIDIA\' not in renderer or \'T4\' not in renderer:\n                raise RuntimeError(\'OpenGL T4 no confirmado; revisar Xorg.log.\')\n            vulkan = capture(as_user([\'vulkaninfo\', \'--summary\']), env=env, timeout=30)\n            (BASE / \'vulkan.log\').write_text(vulkan)\n            if not re.search(r\'deviceName\\s*=\\s*[^\\n]*T4\', vulkan):\n                raise RuntimeError(\'Vulkan64 T4 no confirmado; revisar vulkan.log e ICD aislado.\')\n            session[\'graphics\'] = {\'opengl_nvidia_t4\': True, \'vulkan_nvidia_t4\': True,\n                \'gles_profile\': next((line.strip() for line in renderer.splitlines()\n                                      if \'OpenGL ES profile version string\' in line), \'No informado por glxinfo\')}\n            audio_env = dict(env); audio_env.pop(\'PULSE_SERVER\', None)\n            pulse = spawn([\'pulseaudio\', \'--daemonize=no\', \'--exit-idle-time=-1\', \'--disallow-exit\',\n                           \'--log-target=stderr\'], \'pulse\', audio_env, processes)\n            def audio_ready():\n                return subprocess.run(as_user([\'pactl\', \'info\']), env=env, stdout=subprocess.DEVNULL,\n                                      stderr=subprocess.DEVNULL, timeout=3).returncode == 0\n            wait_for(audio_ready, \'PulseAudio\', [pulse])\n            sinks = capture(as_user([\'pactl\', \'list\', \'short\', \'sinks\']), env=env)\n            if not any(line.split()[1] == \'output\' for line in sinks.splitlines() if len(line.split()) >= 2):\n                capture(as_user([\'pactl\', \'load-module\', \'module-null-sink\', \'sink_name=output\',\n                                 \'sink_properties=device.description=PC-Free\']), env=env)\n            capture(as_user([\'pactl\', \'set-default-sink\', \'output\']), env=env)\n            capture(as_user([\'pactl\', \'set-default-source\', \'output.monitor\']), env=env)\n            dbus = spawn([\'dbus-daemon\', \'--session\', \'--nofork\',\n                          \'--address=\' + env[\'DBUS_SESSION_BUS_ADDRESS\']], \'dbus\', env, processes)\n            def bus_ready():\n                return subprocess.run(as_user([\'dbus-send\', \'--session\', \'--print-reply\',\n                                      \'--dest=org.freedesktop.DBus\', \'/org/freedesktop/DBus\',\n                                      \'org.freedesktop.DBus.ListNames\']), env=env,\n                                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\n                                      timeout=3).returncode == 0\n            wait_for(bus_ready, \'D-Bus de escritorio\', [dbus])\n            desktop = spawn([\'xfce4-session\'], \'xfce\', env, processes)\n            def desktop_ready():\n                result = subprocess.run(as_user([\'xprop\', \'-root\', \'_NET_SUPPORTING_WM_CHECK\']),\n                                        env=env, capture_output=True, text=True, timeout=3)\n                return result.returncode == 0 and \'window id #\' in result.stdout\n            wait_for(desktop_ready, \'XFCE\', [desktop])\n        with phase(\'Selkies autenticado y túnel HTTPS HTTP/2\'):\n            access = dict(username=\'pcfree\', password=secrets.token_urlsafe(24), url=None)\n            private_json(BASE / \'access.json\', access)\n            server_env = dict(env, SELKIES_BASIC_AUTH_USER=access[\'username\'],\n                              SELKIES_BASIC_AUTH_PASSWORD=access[\'password\'],\n                              SELKIES_KEYBOARD_SHORTCUTS=\'true\',\n                              SELKIES_UI_SIDEBAR_SHOW_GAMING_MODE=\'true\',\n                              SELKIES_UI_SIDEBAR_SHOW_FULLSCREEN=\'true\',\n                              SELKIES_UI_SIDEBAR_SHOW_KEYBOARD_BUTTON=\'true\',\n                              SELKIES_VIDEO_STREAMING_MODE=\'true|locked\',\n                              SELKIES_USE_CPU=\'false|locked\', SELKIES_AUTO_GPU=\'nvidia\',\n                              SELKIES_RATE_CONTROL_MODE=\'cbr\')\n            server_env.pop(\'LD_PRELOAD\', None)\n            server = spawn([\'/opt/selkies/bin/selkies\', \'--addr=127.0.0.1\', f\'--port={PORT}\',\n                            \'--web-root=\' + web_root,\n                            \'--mode=websockets\', \'--enable-basic-auth=true\', \'--enable-https=false\',\n                            \'--encoder=h264enc\', f\'--framerate={framerate}\', f\'--video-bitrate={bitrate}\',\n                            \'--enable-resize=false\', \'--backpressure-queue-size=2\'], \'selkies\', server_env, processes)\n            wait_for(lambda: auth_ready(access[\'username\'], access[\'password\']), \'Selkies + autenticación\', [server], 60)\n            cloud = spawn([str(tunnel), \'tunnel\', \'--url\', f\'http://127.0.0.1:{PORT}\',\n                           \'--protocol\', \'http2\', \'--no-autoupdate\'], \'cloudflared\', env, processes)\n            def tunnel_ready():\n                log = (BASE / \'cloudflared.log\').read_text()\n                links = re.findall(r\'https://[a-z0-9-]+\\.trycloudflare\\.com\', log)\n                if links and \'Registered tunnel connection\' in log and \'protocol=http2\' in log:\n                    access[\'url\'] = links[-1]\n                    return True\n                return False\n            # Do NOT require VM DNS/outbound reachability of the quick-tunnel URL.\n            wait_for(tunnel_ready, \'Cloudflare HTTP/2\', [cloud, server], 90)\n            private_json(BASE / \'access.json\', access)\n        session[\'processes\'] = {name: {\'pid\': process.pid, \'start\': pid_identity(process.pid)}\n                                for name, process in processes.items()}\n        private_json(state, session)\n        print(f\'Escritorio preparado en {time.monotonic() - started:.1f} s.\', flush=True)\n        show_access()\n        if firefox:\n            firefox_background(session)\n        return session\n\n\nif __name__ == \'__main__\':\n    bootstrap()\n', 'compat-steam.py': '#!/usr/bin/env python3\n"""Steam oficial opcional; instala, pero no abre ventanas.\n\nAPI: install_steam(session), install_gamepads(session) (sin Steam).\nEstado: /tmp/pcfree-selkies/session.json.\nOpciones en session[\'steam\']: gamepads=False, repair_client=False,\n    bwrap=None (ruta a un proot-bwrap ya instalado y revisado).\nNo cambia el controlador del kernel, libudev del sistema ni el Python global.\n\nFuentes: repo.steampowered.com/steam/archive/stable/steam_latest.deb;\ngithub.com/selkies-project/selkies/tree/2.0.0/addons/fake-udev;\ngithub.com/selkies-project/proot-bwrap (alternativa, no frontera de seguridad).\nfake-udev sirve para mandos; no es una reparación demostrada del login.\n"""\n\nimport argparse\nimport datetime\nimport json\nimport os\nfrom pathlib import Path\nimport pwd\nimport re\nimport shlex\nimport shutil\nimport struct\nimport subprocess\nimport tarfile\nimport tempfile\nimport urllib.parse\nimport urllib.request\n\n\nSTEAM_DEB = \'https://repo.steampowered.com/steam/archive/stable/steam_latest.deb\'\nSELKIES_TAG = \'2.0.0\'\nTRIPLETS = {\'amd64\': \'x86_64-linux-gnu\', \'i386\': \'i386-linux-gnu\'}\n\n\ndef _run(argv, *, cwd=None, log=None, timeout=600, env=None, host_gpu=False):\n    """APT/build sin heredar preloads del escritorio ni imprimir credenciales."""\n    child_env = dict(os.environ)\n    child_env.update(env or {})\n    child_env.update(DEBIAN_FRONTEND=\'noninteractive\', LC_ALL=\'C\')\n    for key in (\'LD_PRELOAD\', \'PYTHONHOME\', \'PYTHONPATH\'):\n        child_env.pop(key, None)\n    if host_gpu:\n        # Sólo nvidia-smi necesita NVML del host Colab; APT/build quedan limpios.\n        if Path(argv[0]).name != \'nvidia-smi\':\n            raise ValueError(\'host_gpu sólo está permitido para nvidia-smi.\')\n        paths = [part for part in child_env.get(\'LD_LIBRARY_PATH\', \'\').split(\':\') if part]\n        paths += [part for part in os.environ.get(\'LD_LIBRARY_PATH\', \'\').split(\':\') if part]\n        child_env[\'LD_LIBRARY_PATH\'] = \':\'.join(dict.fromkeys(paths + [\'/usr/lib64-nvidia\']))\n    else:\n        child_env.pop(\'LD_LIBRARY_PATH\', None)\n    if argv[0] == \'apt-get\':\n        argv = [argv[0], \'-o\', \'APT::Get::AllowUnauthenticated=false\',\n                \'-o\', \'Acquire::AllowInsecureRepositories=false\',\n                \'-o\', \'Acquire::AllowDowngradeToInsecureRepositories=false\', *argv[1:]]\n    result = subprocess.run(argv, cwd=cwd, env=child_env, text=True,\n                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n                            timeout=timeout)\n    if log:\n        with Path(log).open(\'a\') as stream:\n            stream.write(\'$ \' + shlex.join(map(str, argv)) + \'\\n\' + result.stdout + \'\\n\')\n    if result.returncode:\n        raise RuntimeError(f"Falló {argv[0]} ({result.returncode}). "\n                           f"Revisar {log or \'la salida\'}: {result.stdout[-1800:]}")\n    return result.stdout.strip()\n\n\ndef _official_apt_uri(uri):\n    """Mismos orígenes que el núcleo; /./ de APT es válido, /../ no."""\n    parsed = urllib.parse.urlparse(uri)\n    if (parsed.username or parsed.password or parsed.port not in (None, 80, 443)\n            or parsed.query or parsed.fragment):\n        return False\n    path = urllib.parse.unquote(parsed.path)\n    if \'..\' in path.split(\'/\'):\n        return False\n    ubuntu = (parsed.scheme in (\'http\', \'https\') and\n              (parsed.hostname in (\'archive.ubuntu.com\', \'security.ubuntu.com\', \'ports.ubuntu.com\')\n               or (parsed.hostname or \'\').endswith(\'.archive.ubuntu.com\')))\n    nvidia = (parsed.scheme == \'https\' and parsed.port in (None, 443) and\n              parsed.hostname == \'developer.download.nvidia.com\' and\n              path.startswith(\'/compute/cuda/repos/ubuntu2404/x86_64/\'))\n    return ubuntu or nvidia\n\n\ndef _safe_install(packages, log):\n    plan = _run([\'apt-get\', \'-s\', \'install\', \'-y\', \'--no-install-recommends\', *packages], log=log)\n    for line in plan.splitlines():\n        if line.startswith(\'Remv \'):\n            raise RuntimeError(\'APT propone eliminar paquetes; operación cancelada.\')\n        if line.startswith(\'Inst \'):\n            name = line.split()[1].split(\':\')[0]\n            if (\'nvidia\' in name or name.startswith((\'linux-image\', \'linux-modules\',\n                                                   \'linux-headers\', \'linux-generic\', \'linux-virtual\'))):\n                raise RuntimeError(f\'APT propone cambiar {name}; no se modifica el kernel/GPU del host.\')\n    origins = _run([\'apt-get\', \'--print-uris\', \'--download-only\', \'install\', \'-y\',\n                    \'--no-install-recommends\', *packages], log=log)\n    for uri in re.findall(r"^\'([^\']*)\'", origins, re.MULTILINE):\n        if not _official_apt_uri(uri):\n            raise RuntimeError(\'APT propone un origen no oficial; operación cancelada.\')\n    _run([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', *packages], log=log)\n\n\ndef _download(url, target):\n    # HTTPS, hosts oficiales; nunca ejecutar un instalador remoto mediante pipe.\n    with urllib.request.urlopen(url, timeout=90) as response:\n        if urllib.parse.urlparse(response.url).hostname not in {\n                \'repo.steampowered.com\', \'github.com\', \'codeload.github.com\'}:\n            raise RuntimeError(\'Redirección de descarga fuera de los hosts oficiales.\')\n        with Path(target).open(\'wb\') as stream:\n            shutil.copyfileobj(response, stream)\n\n\ndef _elf(path, arch):\n    with Path(path).open(\'rb\') as stream:\n        head = stream.read(20)\n    expected = (2, 62) if arch == \'amd64\' else (1, 3)\n    if (len(head) != 20 or head[:4] != b\'\\x7fELF\' or head[5] != 1 or\n            (head[4], struct.unpack(\'<H\', head[18:20])[0]) != expected):\n        raise RuntimeError(f\'Biblioteca ELF incompatible ({arch}): {path}\')\n\n\ndef _owned_write(path, text, account, executable=False):\n    path = Path(path)\n    _user_directory(path.parent, account)\n    path.write_text(text)\n    path.chmod(0o755 if executable else 0o644)\n    os.chown(path, account.pw_uid, account.pw_gid)\n\n\ndef _user_directory(path, account):\n    missing = []\n    cursor = Path(path)\n    while not cursor.exists():\n        missing.append(cursor)\n        cursor = cursor.parent\n    for directory in reversed(missing):\n        directory.mkdir(mode=0o755)\n        os.chown(directory, account.pw_uid, account.pw_gid)\n\n\ndef _exact_package(package, arch, driver, log):\n    lines = _run([\'apt-cache\', \'madison\', f\'{package}:{arch}\'], log=log)\n    versions = [line.split(\'|\')[1].strip() for line in lines.splitlines()\n                if len(line.split(\'|\')) >= 3]\n    candidates = [v for v in versions if v.split(\':\')[-1].split(\'-\')[0] == driver]\n    if not candidates:\n        raise RuntimeError(f\'Falta {package}:{arch} para NVIDIA {driver} en APT. \'\n                           \'Habilitar un repositorio oficial Ubuntu/NVIDIA con esa versión; \'\n                           \'no instalar otro controlador ni sustituir por una versión distinta.\')\n    # APT ordena sus resultados; no asumir que la revisión Ubuntu es la de otra arquitectura.\n    return candidates[0]\n\n\ndef _graphics(base, driver, log):\n    """Extrae sólo GL/gpucomp; no instala metapaquetes NVIDIA, DKMS o módulos."""\n    branch = driver.split(\'.\')[0]\n    root = base / \'nvidia\' / driver\n    root.mkdir(parents=True, exist_ok=True)\n    specs = []\n    for arch in TRIPLETS:\n        package = f\'libnvidia-gl-{branch}\'\n        version = _exact_package(package, arch, driver, log)\n        specs.append((package, arch, version))\n        # Algunas ramas separan gpucomp; leer la dependencia de la versión exacta.\n        metadata = _run([\'apt-cache\', \'show\', f\'{package}:{arch}={version}\'], log=log)\n        dependencies = re.findall(r\'libnvidia-gpucomp-[0-9]+\', metadata)\n        for dependency in sorted(set(dependencies)):\n            specs.append((dependency, arch,\n                          _exact_package(dependency, arch, driver, log)))\n    # Resolver todos los orígenes ANTES de descargar; APT verifica hashes contra\n    # metadatos autenticados. No añadir claves, trusted=yes ni PPAs.\n    for package, arch, version in specs:\n        origins = _run([\'apt-get\', \'--print-uris\', \'download\', f\'{package}:{arch}={version}\'], log=log)\n        uris = re.findall(r"^\'([^\']+)\'", origins, re.MULTILINE)\n        if not uris or not all(_official_apt_uri(uri) for uri in uris):\n            raise RuntimeError(f\'{package}:{arch}: origen NVIDIA/Ubuntu no permitido.\')\n    with tempfile.TemporaryDirectory(prefix=\'download-\', dir=base) as temporary:\n        for package, arch, version in specs:\n            _run([\'apt-get\', \'download\', f\'{package}:{arch}={version}\'],\n                 cwd=temporary, log=log)\n        for deb in sorted(Path(temporary).glob(\'*.deb\')):\n            _run([\'dpkg-deb\', \'-x\', str(deb), str(root)], log=log)\n    directories = []\n    for arch, triplet in TRIPLETS.items():\n        directory = root / \'usr/lib\' / triplet\n        for library in (\'libGLX_nvidia.so.0\', \'libEGL_nvidia.so.0\'):\n            _elf(directory / library, arch)\n        directories.append(str(directory))\n    # El mismo nombre se resuelve con el ELF correcto en cada proceso. No fijar\n    # una ruta absoluta amd64 en el ICD que también utilizará Steam de 32 bits.\n    icd = root / \'share/vulkan/icd.d/nvidia.json\'\n    icd.parent.mkdir(parents=True, exist_ok=True)\n    icd.write_text(json.dumps({\'file_format_version\': \'1.0.0\', \'ICD\': {\n        \'library_path\': \'libGLX_nvidia.so.0\', \'api_version\': \'1.3.0\'}}, indent=2))\n    egl = root / \'share/glvnd/egl_vendor.d/nvidia.json\'\n    egl.parent.mkdir(parents=True, exist_ok=True)\n    egl.write_text(json.dumps({\'file_format_version\': \'1.0.0\', \'ICD\': {\n        \'library_path\': \'libEGL_nvidia.so.0\'}}, indent=2))\n    return {\'LD_LIBRARY_PATH\': \':\'.join(directories),\n            \'__GLX_VENDOR_LIBRARY_NAME\': \'nvidia\',\n            \'__EGL_VENDOR_LIBRARY_FILENAMES\': str(egl),\n            \'VK_DRIVER_FILES\': str(icd), \'VK_ICD_FILENAMES\': str(icd)}, specs\n\n\ndef _gamepads(base, log, shared_root):\n    """fake-udev privado, 2 ELF verificados. Nunca sustituye libudev del sistema."""\n    _safe_install([\'build-essential\', \'gcc-multilib\', \'libc6-dev-i386\',\n                   \'libudev1:amd64\', \'libudev1:i386\'], log)\n    with tempfile.TemporaryDirectory(prefix=\'fake-udev-\', dir=base) as temporary:\n        archive = Path(temporary) / \'selkies.tar.gz\'\n        _download(f\'https://github.com/selkies-project/selkies/archive/refs/tags/{SELKIES_TAG}.tar.gz\',\n                  archive)\n        with tarfile.open(archive) as source:\n            # Extraer sólo este addon y rechazar enlaces/path traversal del archivo.\n            prefix = f\'selkies-{SELKIES_TAG}/addons/fake-udev/\'\n            members = [member for member in source.getmembers()\n                       if member.name.startswith(prefix)]\n            if not members:\n                raise RuntimeError(\'El archivo oficial no contiene fake-udev.\')\n            for member in members:\n                if (\'..\' in Path(member.name).parts or\n                        not (member.isfile() or member.isdir())):\n                    raise RuntimeError(\'Entrada de archivo no segura en fake-udev.\')\n            source.extractall(temporary, members=members, filter=\'data\')\n        build = Path(temporary) / prefix\n        _run([\'make\', \'-C\', str(build), \'all\', \'all32\'], log=log)\n        for arch, triplet in TRIPLETS.items():\n            filename = \'libudev.so.1.0.0-fake\' if arch == \'amd64\' else \'libudev_x86.so.1.0.0-fake\'\n            source = build / filename\n            _elf(source, arch)\n            destination = shared_root / \'lib\' / triplet\n            destination.mkdir(parents=True, exist_ok=True)\n            # Reinstalar no debe truncar una biblioteca ya mapeada por un juego.\n            staged = destination / \'libudev.so.1.new\'\n            shutil.copy2(source, staged)\n            staged.replace(destination / \'libudev.so.1\')\n            # La .deb oficial Selkies proporciona ambas variantes del interposer.\n            interposer = Path(\'/usr/lib\') / triplet / \'selkies_input_interposer.so\'\n            _elf(interposer, arch)\n    input_directory = Path(\'/dev/input\')\n    if not input_directory.exists():\n        input_directory.mkdir(mode=0o1777)\n        input_directory.chmod(0o1777)\n    # $LIB es literal: glibc lo expande según la arquitectura DEL PROCESO.\n    library = str(shared_root / \'$LIB\')\n    return {\'SELKIES_INTERPOSER\': \'/usr/$LIB/selkies_input_interposer.so\',\n            \'fake_udev\': library + \'/libudev.so.1\'}\n\n\ndef _quarantine_client(home, base):\n    """Reparación solicitada: sólo carpetas redistribuibles, jamás cuentas/juegos."""\n    client = home / \'.local/share/Steam\'\n    if client.is_symlink():\n        raise RuntimeError(\'Steam usa un enlace externo; revisar manualmente antes de reparar.\')\n    quarantine = base / (\'quarantine-\' + datetime.datetime.now(datetime.timezone.utc)\n                         .strftime(\'%Y%m%dT%H%M%S%fZ\'))\n    for name in (\'ubuntu12_32\', \'ubuntu12_64\', \'steam.sh\'):\n        target = client / name\n        if target.exists() or target.is_symlink():\n            quarantine.mkdir(exist_ok=True)\n            target.rename(quarantine / name)\n    return str(quarantine) if quarantine.exists() else None\n\n\ndef install_gamepads(session):\n    """Mandos opcionales independientes de Steam; no lanza/reinicia procesos."""\n    if os.geteuid() != 0:\n        raise RuntimeError(\'La instalación de mandos requiere root; las aplicaciones no.\')\n    account = pwd.getpwnam(session[\'user\'])\n    if account.pw_uid == 0:\n        raise ValueError(\'Los juegos deben ejecutarse como usuario normal.\')\n    home, runtime = Path(session[\'home\']), Path(session[\'runtime\'])\n    if not home.is_absolute() or not runtime.is_absolute() or not home.is_dir() or not runtime.is_dir():\n        raise ValueError(\'Preparar primero home y runtime absolutos del escritorio.\')\n    shared_root = Path(session.get(\'fake_udev_root\', runtime.parent / \'fake-udev\'))\n    if not shared_root.is_absolute():\n        raise ValueError(\'fake_udev_root debe ser una ruta absoluta.\')\n    base = home / \'.local/share/pcfree/gamepads\'\n    _user_directory(base, account)\n    log = base / \'install.log\'\n    _run([\'dpkg\', \'--add-architecture\', \'i386\'], log=log)\n    _run([\'apt-get\', \'update\', \'-qq\'], log=log)\n    preload = _gamepads(base, log, shared_root)\n    result = {\'installed\': True, \'root\': str(shared_root), \'log\': str(log),\n              \'source_tag\': SELKIES_TAG, \'architectures\': list(TRIPLETS),\n              \'env\': {\'SELKIES_INTERPOSER\': preload[\'SELKIES_INTERPOSER\'],\n                      \'LD_PRELOAD\': preload[\'SELKIES_INTERPOSER\'] + \':\' + preload[\'fake_udev\']},\n              \'unset_env\': [\'SDL_JOYSTICK_DISABLE_UDEV\', \'SDL_JOYSTICK_DEVICE\']}\n    _owned_write(base / \'installation.json\', json.dumps(result, indent=2), account)\n    print(\'Mandos: fake-udev aislado e interposer oficial verificados para 32 y 64 bits.\')\n    print(\'Aplicar sólo a aplicaciones/juegos; no precargar en el servidor Selkies.\')\n    return result\n\n\ndef _launch_script(client, graphics, gamepads, bwrap):\n    assignments = \'\'.join(\'export \' + key + \'=\' + shlex.quote(value) + \'\\n\'\n                          for key, value in graphics.items() if key != \'LD_LIBRARY_PATH\')\n    assignments += (\'export LD_LIBRARY_PATH=\' + shlex.quote(graphics[\'LD_LIBRARY_PATH\']) +\n                    \'${LD_LIBRARY_PATH:+:$LD_LIBRARY_PATH}\\n\')\n    if gamepads:\n        assignments += \'unset SDL_JOYSTICK_DISABLE_UDEV SDL_JOYSTICK_DEVICE\\n\'\n        # Reusar el input interposer oficial heredado de pcfree-run; copiarlo\n        # a otra ruta podría cargar dos instancias de los mismos hooks.\n        # No usar expansión shell de $LIB en este valor por defecto.\n        assignments += (\'if [ -z "${SELKIES_INTERPOSER:-}" ]; then export SELKIES_INTERPOSER=\' +\n                        shlex.quote(gamepads[\'SELKIES_INTERPOSER\']) + \'; fi\\n\')\n        assignments += \'\'\'prepend_preload() {\n  case ":${LD_PRELOAD:-}:" in *":$1:"*) return;; esac\n  LD_PRELOAD="$1${LD_PRELOAD:+:$LD_PRELOAD}"\n  export LD_PRELOAD\n}\nprepend_preload "$SELKIES_INTERPOSER"\n\'\'\'\n        assignments += \'prepend_preload \' + shlex.quote(gamepads[\'fake_udev\']) + \'\\n\'\n    if bwrap:\n        assignments += \'export BWRAP=\' + shlex.quote(bwrap) + \'\\n\'\n    return \'\'\'#!/bin/sh\nset -eu\nif [ "$(id -u)" = 0 ]; then echo \'Steam debe ejecutarse como usuario normal.\' >&2; exit 1; fi\n: "${HOME:?}" "${XDG_RUNTIME_DIR:?}" "${DISPLAY:?}" "${XAUTHORITY:?}"\nbus_ok() {\n  dbus-send --bus="$1" --type=method_call --print-reply --dest=org.freedesktop.DBus \\\n    /org/freedesktop/DBus org.freedesktop.DBus.ListNames >/dev/null 2>&1\n}\n# Preferir el bus persistente del escritorio. Nunca dbus-run-session alrededor\n# del bootstrap: éste puede terminar antes que su steamwebhelper desacoplado.\nif [ -z "${DBUS_SESSION_BUS_ADDRESS:-}" ] || ! bus_ok "$DBUS_SESSION_BUS_ADDRESS"; then\n  address="unix:path=$XDG_RUNTIME_DIR/steam-session-bus"\n  (\n    flock -x 9\n    if ! bus_ok "$address"; then\n      dbus-daemon --session --fork --address="$address"\n    fi\n  ) 9>"$XDG_RUNTIME_DIR/steam-session-bus.lock"\n  export DBUS_SESSION_BUS_ADDRESS="$address"\nfi\n\'\'\' + assignments + \'exec \' + shlex.quote(str(client)) + \' "$@"\\n\'\n\n\ndef install_steam(session):\n    """Instalación explícita opcional, sin lanzar Steam ni modificar el estado global."""\n    if os.geteuid() != 0:\n        raise RuntimeError(\'Ejecutar la celda instaladora como root; Steam abrirá como usuario normal.\')\n    required = (\'user\', \'home\', \'runtime\', \'display\', \'xauthority\', \'gpu_root\',\n                \'driver_version\', \'env\', \'run_launcher\')\n    missing = [key for key in required if key not in session]\n    if missing:\n        raise ValueError(\'Estado incompleto: \' + \', \'.join(missing))\n    account = pwd.getpwnam(session[\'user\'])\n    if account.pw_uid == 0:\n        raise ValueError(\'El usuario de Steam no puede ser root.\')\n    home = Path(session[\'home\'])\n    runtime = Path(session[\'runtime\'])\n    runner = Path(session[\'run_launcher\'])\n    if not all(path.is_absolute() for path in (home, runtime, runner, Path(session[\'gpu_root\']))):\n        raise ValueError(\'El estado debe proporcionar rutas absolutas.\')\n    if not home.is_dir() or not runtime.is_dir() or not os.access(runner, os.X_OK):\n        raise RuntimeError(\'Preparar primero el escritorio y pcfree-run.\')\n    for key, value in {\'HOME\': str(home), \'USER\': session[\'user\'],\n                       \'DISPLAY\': session[\'display\'], \'XAUTHORITY\': session[\'xauthority\'],\n                       \'XDG_RUNTIME_DIR\': str(runtime)}.items():\n        if session[\'env\'].get(key) != value:\n            raise ValueError(f\'Estado y env no coinciden en {key}.\')\n    driver = session[\'driver_version\']\n    if not re.fullmatch(r\'\\d+\\.\\d+(?:\\.\\d+)?\', driver):\n        raise ValueError(\'Versión NVIDIA no válida.\')\n    active = _run([\'nvidia-smi\', \'--query-gpu=driver_version\', \'--format=csv,noheader\'],\n                  env={\'LD_LIBRARY_PATH\': session[\'env\'].get(\'LD_LIBRARY_PATH\', \'\')}, host_gpu=True)\n    if not active.splitlines() or any(v.strip() != driver for v in active.splitlines()):\n        raise RuntimeError(\'El controlador activo no coincide con session.json; no se cambiará.\')\n    options = session.get(\'steam\', {})\n    if not isinstance(options, dict):\n        raise ValueError(\'session[steam] debe ser un objeto.\')\n    bwrap = options.get(\'bwrap\')\n    if bwrap and (not Path(bwrap).is_absolute() or not os.access(bwrap, os.X_OK)):\n        raise ValueError(\'bwrap debe ser una ruta absoluta a un instalador revisado ya ejecutable.\')\n    base = home / \'.local/share/pcfree/steam\'\n    _user_directory(base, account)\n    log = base / \'install.log\'\n    _run([\'dpkg\', \'--add-architecture\', \'i386\'], log=log)\n    _run([\'apt-get\', \'update\', \'-qq\'], log=log)\n    packages = [\'ca-certificates\', \'dbus\', \'dbus-x11\', \'util-linux\',\n                \'lib32gcc-s1\', \'lib32stdc++6\']\n    # Ubuntu 24.04+ usa libasound2t64; no solicitar el paquete virtual libasound2.\n    asound = (\'libasound2t64\' if _run([\'apt-cache\', \'search\', \'^libasound2t64$\'])\n              else \'libasound2\')\n    for arch in TRIPLETS:\n        packages += [f\'{name}:{arch}\' for name in (\n            \'libc6\', \'libstdc++6\', \'libgcc-s1\', \'libgl1\', \'libegl1\', \'libgles2\',\n            \'libvulkan1\', \'libdrm2\', \'libx11-6\', \'libxext6\', \'libxrandr2\',\n            \'libxinerama1\', \'libxcursor1\', \'libxi6\', \'libxss1\', \'libxtst6\',\n            \'libxkbcommon-x11-0\', \'libudev1\', \'libdbus-1-3\', \'libpulse0\',\n            asound, \'libasound2-plugins\', \'libfontconfig1\', \'libfreetype6\')]\n    _safe_install(packages, log)\n    graphics, specs = _graphics(base, driver, log)\n    fake_root = Path(session.get(\'fake_udev_root\', runtime.parent / \'fake-udev\'))\n    if not fake_root.is_absolute():\n        raise ValueError(\'fake_udev_root debe ser una ruta absoluta.\')\n    gamepads = _gamepads(base, log, fake_root) if options.get(\'gamepads\', False) else {}\n    deb = base / \'steam_latest.deb\'\n    _download(STEAM_DEB, deb)\n    version = _run([\'dpkg-deb\', \'-f\', str(deb), \'Version\'], log=log)\n    if _run([\'dpkg-deb\', \'-f\', str(deb), \'Package\'], log=log) != \'steam-launcher\':\n        raise RuntimeError(\'La descarga oficial no contiene steam-launcher.\')\n    with tempfile.TemporaryDirectory(prefix=\'launcher-\', dir=base) as temporary:\n        extracted = Path(temporary) / \'root\'\n        _run([\'dpkg-deb\', \'-x\', str(deb), str(extracted)], log=log)\n        if (not (extracted / \'usr/lib/steam/bin_steam.sh\').is_file() or\n                not (extracted / \'usr/lib/steam/bootstraplinux_ubuntu12_32.tar.xz\').is_file()):\n            raise RuntimeError(\'Cambió el diseño del paquete oficial; revisar el bootstrap.\')\n        destination = base / \'launcher\'\n        if destination.exists():\n            destination.rename(base / (\'launcher-old-\' + datetime.datetime.now(datetime.timezone.utc)\n                                       .strftime(\'%Y%m%dT%H%M%S%fZ\')))\n        extracted.rename(destination)\n    quarantine = None\n    if options.get(\'repair_client\', False):\n        # No terminar procesos de una sesión activa, ni mover archivos en uso.\n        running = subprocess.run([\'pgrep\', \'-u\', str(account.pw_uid), \'-x\', \'steam\'],\n                                 stdout=subprocess.DEVNULL, check=False)\n        if running.returncode == 0:\n            raise RuntimeError(\'Cerrar Steam manualmente antes de repair_client; no se ha detenido.\')\n        quarantine = _quarantine_client(home, base)\n    # Su nombre debe ser steam: algunos bootstrap consultan basename($0).\n    client = destination / \'usr/lib/steam/steam\'\n    if not client.exists():\n        client.symlink_to(\'bin_steam.sh\')\n    launch = base / \'start-steam.sh\'\n    _owned_write(launch, _launch_script(client, graphics, gamepads, bwrap), account, True)\n    command = shlex.join([str(runner), str(launch)])\n    wrapper = home / \'.local/bin/pcfree-steam\'\n    _owned_write(wrapper, \'#!/bin/sh\\nexec \' + command + \' "$@"\\n\', account, True)\n    desktop = \'[Desktop Entry]\\nType=Application\\nName=Steam\\nComment=Cliente oficial (inicio manual)\\n\'\n    # Desktop Entry Exec usa comillas dobles, no el quoting de un shell.\n    quote = lambda value: \'"\' + str(value).replace(\'\\\\\', \'\\\\\\\\\').replace(\'"\', \'\\\\"\').replace(\'`\', \'\\\\`\').replace(\'$\', \'\\\\$\') + \'"\'\n    desktop += (\'Exec=\' + quote(runner) + \' \' + quote(launch) + \'\\n\'\n                \'Icon=\' + str(destination / \'usr/share/icons/hicolor/256x256/apps/steam.png\') + \'\\n\'\n                \'Terminal=false\\nCategories=Game;\\n\')\n    _owned_write(home / \'.local/share/applications/pcfree-steam.desktop\', desktop, account)\n    if (home / \'Desktop\').is_dir():\n        _owned_write(home / \'Desktop/Steam.desktop\', desktop, account, True)\n    # El usuario puede abrir desde el menú; no confiar en metadatos GVFS root.\n    result = {\'installed\': True, \'login_validated\': False, \'version\': version,\n              \'command\': command, \'log\': str(log), \'driver_version\': driver,\n              \'nvidia_packages\': specs, \'gamepads\': bool(gamepads),\n              \'quarantine\': quarantine, \'bwrap\': bwrap}\n    _owned_write(base / \'installation.json\', json.dumps(result, indent=2), account)\n    # Todo el árbol privado debe ser modificable por el usuario del cliente.\n    for directory, directories, files in os.walk(base):\n        os.chown(directory, account.pw_uid, account.pw_gid)\n        for name in directories + files:\n            os.chown(Path(directory) / name, account.pw_uid, account.pw_gid,\n                     follow_symlinks=False)\n    print(\'Steam oficial instalado; no se ha abierto. Abrir Steam desde el menú.\')\n    print(\'El primer inicio descarga el cliente. Login y juegos requieren prueba manual.\')\n    print(\'Si pressure-vessel rechaza namespaces, revisar proot-bwrap de Selkies; no desactivar seccomp globalmente.\')\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--session\', default=\'/tmp/pcfree-selkies/session.json\')\n    parser.add_argument(\'--gamepads\', action=\'store_true\', help=\'Compilar fake-udev oficial aislado (ambos ELF).\')\n    parser.add_argument(\'--gamepads-only\', action=\'store_true\', help=\'Instalar sólo mandos, sin descargar Steam.\')\n    parser.add_argument(\'--repair-client\', action=\'store_true\', help=\'Poner sólo binarios redistribuibles en cuarentena.\')\n    args = parser.parse_args()\n    session = json.loads(Path(args.session).read_text())\n    session[\'steam\'] = dict(session.get(\'steam\', {}))\n    if args.gamepads:\n        session[\'steam\'][\'gamepads\'] = True\n    if args.repair_client:\n        session[\'steam\'][\'repair_client\'] = True\n    if args.gamepads_only:\n        install_gamepads(session)\n    else:\n        install_steam(session)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'compat-apps.py': '"""Optional applications. Install only; never start graphical applications."""\nimport hashlib\nimport fcntl\nimport json\nimport os\nimport platform\nimport pwd\nimport re\nimport shlex\nimport subprocess\nimport tarfile\nimport threading\nimport urllib.request\nimport urllib.parse\nimport zipfile\nfrom pathlib import Path\n\nBASE = Path(\'/tmp/pcfree-selkies\')\nOPTIONS = {\n    \'wine\': [\'wine64\', \'wine32:i386\', \'winetricks\'],\n    \'lutris\': [\'lutris\'],\n    \'java\': [\'openjdk-21-jre\'],\n    \'gaming-libs\': [\'libsdl2-2.0-0\', \'libsdl2-2.0-0:i386\',\n                    \'libopenal1\', \'libopenal1:i386\', \'libvulkan1:i386\',\n                    \'libasound2t64:i386\', \'libpulse0:i386\', \'libstdc++6:i386\'],\n    \'deb-installer\': [\'gdebi\'],\n    \'blender\': [\'blender\'],\n}\nAPPIMAGES = {\n    \'heroic\': (\'Heroic-Games-Launcher/HeroicGamesLauncher\', \'Heroic\'),\n    \'prism\': (\'PrismLauncher/PrismLauncher\', \'Prism Launcher\'),\n}\nEXPERIMENTAL = {\'godot\', \'vinegar\', \'roblox\'}\n\n\ndef command(argv, log, timeout=600):\n    env = {**os.environ, \'DEBIAN_FRONTEND\': \'noninteractive\', \'LC_ALL\': \'C\'}\n    for key in (\'LD_PRELOAD\', \'LD_LIBRARY_PATH\', \'PYTHONPATH\', \'PYTHONHOME\'):\n        env.pop(key, None)\n    if argv[:2] == [\'apt-get\', \'install\']:\n        plan = subprocess.check_output([\'apt-get\', \'-s\', *argv[1:]], env=env,\n                                       text=True, stderr=subprocess.STDOUT, timeout=120)\n        for line in plan.splitlines():\n            if line.startswith(\'Remv \'):\n                raise RuntimeError(\'APT propone eliminar paquetes; se canceló la instalación.\')\n            if line.startswith(\'Inst \'):\n                package = line.split()[1].split(\':\')[0]\n                if \'nvidia\' in package or package.startswith((\'linux-image\', \'linux-modules\', \'linux-headers\')):\n                    raise RuntimeError(\'APT propone modificar GPU/kernel; se canceló la instalación.\')\n        origins = subprocess.check_output([\'apt-get\', \'--print-uris\', \'--download-only\', *argv[1:]],\n                                          env=env, text=True, stderr=subprocess.STDOUT, timeout=120)\n        for uri in re.findall(r"^\'([^\']+)\'", origins, re.MULTILINE):\n            parsed = urllib.parse.urlparse(uri)\n            host = parsed.hostname or \'\'\n            ubuntu = host in {\'archive.ubuntu.com\', \'security.ubuntu.com\', \'ports.ubuntu.com\'} or host.endswith(\'.archive.ubuntu.com\')\n            nvidia = host == \'developer.download.nvidia.com\' and parsed.scheme == \'https\' and parsed.path.startswith(\'/compute/cuda/repos/ubuntu2404/x86_64/\')\n            if parsed.scheme not in {\'http\', \'https\'} or not (ubuntu or nvidia) or parsed.username or parsed.password:\n                raise RuntimeError(\'La instalación requiere paquetes de repositorios oficiales Ubuntu/NVIDIA.\')\n    with log.open(\'a\') as stream:\n        result = subprocess.run(argv, stdout=stream, stderr=subprocess.STDOUT,\n                                env=env,\n                                timeout=timeout)\n    if result.returncode:\n        raise RuntimeError(f\'Falló {shlex.join(argv)}; revisar {log}\\n\'\n                           + log.read_text(errors=\'replace\')[-3000:])\n\n\ndef shortcut(session, name, executable, category=\'Game\', icon=\'applications-games\'):\n    home = Path(session[\'home\'])\n    desktop = home / \'Desktop\'\n    desktop.mkdir(parents=True, exist_ok=True)\n    path = desktop / (name.replace(\' \', \'-\') + \'.desktop\')\n    # Quote Desktop Entry arguments, independently of shell quoting.\n    def quote(arg):\n        return \'"\' + str(arg).replace(\'\\\\\', \'\\\\\\\\\').replace(\'"\', \'\\\\"\').replace(\'`\', \'\\\\`\').replace(\'$\', \'\\\\$\') + \'"\'\n    launcher = session.get(\'run_launcher\', \'/usr/local/bin/pcfree-run\')\n    path.write_text(\'[Desktop Entry]\\nType=Application\\nVersion=1.0\\n\'\n                    f\'Name={name}\\nExec={quote(launcher)} {quote(executable)}\\n\'\n                    f\'Icon={icon}\\nTerminal=false\\nCategories={category};\\n\')\n    path.chmod(0o755)\n    account = pwd.getpwnam(session[\'user\'])\n    os.chown(path, account.pw_uid, account.pw_gid)\n    env = session[\'env\']\n    subprocess.run([\'runuser\', \'-u\', session[\'user\'], \'--\', \'env\',\n                    *[f\'{k}={v}\' for k, v in env.items()],\n                    \'gio\', \'set\', str(path), \'metadata::trusted\', \'true\'],\n                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n\n\ndef official_appimage(option, session):\n    repository, label = APPIMAGES[option]\n    directory = BASE / \'apps\' / option\n    directory.mkdir(parents=True, exist_ok=True)\n    request = urllib.request.Request(\n        f\'https://api.github.com/repos/{repository}/releases/latest\',\n        headers={\'User-Agent\': \'PC-Free-T4\'})\n    with urllib.request.urlopen(request, timeout=45) as response:\n        release = json.load(response)\n    matches = [a for a in release.get(\'assets\', [])\n               if a[\'name\'].lower().endswith(\'.appimage\')\n               and (\'x86_64\' in a[\'name\'].lower() or \'amd64\' in a[\'name\'].lower())\n               and \'portable\' not in a[\'name\'].lower()]\n    if len(matches) != 1:\n        raise RuntimeError(f\'No se pudo elegir el AppImage oficial de {repository}: \'\n                           + \', \'.join(a[\'name\'] for a in matches))\n    asset = matches[0]\n    url = asset[\'browser_download_url\']\n    if not url.startswith(f\'https://github.com/{repository}/releases/download/\'):\n        raise RuntimeError(\'El enlace del paquete no pertenece al repositorio oficial.\')\n    image = directory / Path(asset[\'name\']).name\n    if not image.exists():\n        temporary = image.with_suffix(image.suffix + \'.part\')\n        urllib.request.urlretrieve(url, temporary)\n        digest = asset.get(\'digest\') or \'\'\n        if digest.startswith(\'sha256:\'):\n            sha = hashlib.sha256()\n            with temporary.open(\'rb\') as source:\n                for block in iter(lambda: source.read(1024 * 1024), b\'\'):\n                    sha.update(block)\n            if sha.hexdigest() != digest.partition(\':\')[2]:\n                temporary.unlink()\n                raise RuntimeError(\'La descarga no coincide con el SHA-256 oficial.\')\n        temporary.replace(image)\n    image.chmod(0o755)\n    extracted = directory / \'squashfs-root\'\n    log = directory / \'install.log\'\n    if not (extracted / \'AppRun\').exists():\n        # Native extraction avoids FUSE. Runs only the official installer runtime.\n        with log.open(\'w\') as stream:\n            result = subprocess.run([str(image), \'--appimage-extract\'], cwd=directory,\n                                    stdout=stream, stderr=subprocess.STDOUT, timeout=180)\n        if result.returncode or not (extracted / \'AppRun\').exists():\n            raise RuntimeError(f\'No se pudo extraer el AppImage; revisar {log}.\')\n    shortcut(session, label, str(extracted / \'AppRun\'))\n    print(f\'{label} {release["tag_name"]}: instalado, sin abrir. \'\n          \'Su sandbox y los juegos requieren pruebas en este entorno.\')\n\n\ndef install_godot(session):\n    directory = BASE / \'apps/godot\'\n    directory.mkdir(parents=True, exist_ok=True)\n    repository = \'godotengine/godot\'\n    request = urllib.request.Request(f\'https://api.github.com/repos/{repository}/releases/latest\',\n                                     headers={\'User-Agent\': \'PC-Free-T4\'})\n    with urllib.request.urlopen(request, timeout=45) as response:\n        release = json.load(response)\n    candidates = [a for a in release[\'assets\']\n                  if a[\'name\'].endswith(\'_linux.x86_64.zip\') and \'mono\' not in a[\'name\'].lower()]\n    if len(candidates) != 1:\n        raise RuntimeError(\'No se encontró un binario Linux x86_64 de Godot en su publicación oficial.\')\n    asset = candidates[0]\n    if not asset[\'browser_download_url\'].startswith(f\'https://github.com/{repository}/releases/download/\'):\n        raise RuntimeError(\'Origen de Godot inesperado.\')\n    archive = directory / Path(asset[\'name\']).name\n    if not archive.exists():\n        urllib.request.urlretrieve(asset[\'browser_download_url\'], archive)\n    with zipfile.ZipFile(archive) as bundle:\n        entries = [item for item in bundle.infolist() if not item.is_dir()]\n        if len(entries) != 1 or \'/\' in entries[0].filename or \'\\\\\' in entries[0].filename:\n            raise RuntimeError(\'Estructura del paquete Godot inesperada.\')\n        executable = directory / entries[0].filename\n        with bundle.open(entries[0]) as source, executable.open(\'wb\') as destination:\n            import shutil\n            shutil.copyfileobj(source, destination)\n    executable.chmod(0o755)\n    shortcut(session, \'Godot\', str(executable), \'Development\', \'applications-development\')\n    print(f\'Godot {release["tag_name"]}: instalado, sin abrir. \'\n          \'Vulkan NVIDIA se prepara en el arranque; Compatibility usa OpenGL si el proyecto lo requiere.\')\n\n\ndef install_vinegar(session):\n    """Official native source build; Studio/Wine downloads start only when opened."""\n    directory = BASE / \'apps/vinegar\'\n    directory.mkdir(parents=True, exist_ok=True)\n    log = directory / \'install.log\'\n    packages = [\'build-essential\', \'git\', \'make\', \'pkg-config\', \'gettext\',\n                \'libgtk-4-dev\', \'libadwaita-1-dev\', \'libvulkan-dev\', \'libglib2.0-dev-bin\']\n    command([\'apt-get\', \'update\', \'-qq\'], log)\n    command([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', *packages], log)\n    request = urllib.request.Request(\'https://go.dev/dl/?mode=json\', headers={\'User-Agent\': \'PC-Free-T4\'})\n    with urllib.request.urlopen(request, timeout=45) as response:\n        releases = json.load(response)\n    choices = [release for release in releases if release.get(\'stable\')\n               and tuple(int(n) for n in release[\'version\'][2:].split(\'.\')[:2]) >= (1, 26)]\n    if not choices:\n        raise RuntimeError(\'Vinegar 1.9.4 requiere Go >=1.26; no se encontró un compilador estable oficial.\')\n    toolchain = next(f for f in choices[0][\'files\'] if f[\'os\'] == \'linux\'\n                     and f[\'arch\'] == \'amd64\' and f[\'kind\'] == \'archive\')\n    tools = directory / \'toolchain\'\n    tools.mkdir(exist_ok=True)\n    archive = tools / toolchain[\'filename\']\n    if not (tools / \'go/bin/go\').exists():\n        urllib.request.urlretrieve(\'https://go.dev/dl/\' + toolchain[\'filename\'], archive)\n        sha = hashlib.sha256()\n        with archive.open(\'rb\') as source:\n            for block in iter(lambda: source.read(1024 * 1024), b\'\'):\n                sha.update(block)\n        if sha.hexdigest() != toolchain[\'sha256\']:\n            archive.unlink()\n            raise RuntimeError(\'SHA-256 del compilador Go no válido.\')\n        with tarfile.open(archive) as bundle:\n            bundle.extractall(tools, filter=\'data\')\n        archive.unlink()\n    source = directory / \'source\'\n    if not source.exists():\n        command([\'git\', \'clone\', \'--branch\', \'v1.9.4\', \'--depth\', \'1\',\n                 \'https://github.com/vinegarhq/vinegar.git\', str(source)], log)\n    prefix = directory / \'installed\'\n    env = {**os.environ, \'PATH\': str(tools / \'go/bin\') + \':\' + os.environ.get(\'PATH\', \'\')}\n    env.pop(\'LD_PRELOAD\', None)\n    for argv in [[\'make\', \'-j2\', \'PREFIX=\' + str(prefix)],\n                 [\'make\', \'install\', \'PREFIX=\' + str(prefix)]]:\n        with log.open(\'a\') as stream:\n            result = subprocess.run(argv, cwd=source, env=env, stdout=stream,\n                                    stderr=subprocess.STDOUT, timeout=1200)\n        if result.returncode:\n            raise RuntimeError(f\'Falló la compilación de Vinegar; revisar {log}.\')\n    shortcut(session, \'Roblox Studio - Vinegar\', str(prefix / \'bin/vinegar\'),\n             \'Development\', str(prefix / \'share/icons/hicolor/scalable/apps/org.vinegarhq.Vinegar.svg\'))\n    print(\'Vinegar 1.9.4: instalado desde el código oficial, sin abrir. \'\n          \'Wine y Roblox Studio se descargan al abrirlo. Experimental: Studio no validado en Colab; \'\n          \'Vinegar no ejecuta Roblox Player.\')\n\n\ndef sober_requirements(cpuinfo=None, machine=None, kernel=None):\n    """Validate hardware, not emulate CPU features or bypass kernel restrictions."""\n    machine = machine or platform.machine()\n    kernel = kernel or platform.release()\n    if machine not in {\'x86_64\', \'amd64\'}:\n        raise RuntimeError(\'Sober requiere CPU x86-64; ARM no está soportado.\')\n    match = re.match(r\'(\\d+)\\.(\\d+)\', kernel)\n    if not match or tuple(map(int, match.groups())) < (5, 11):\n        raise RuntimeError(\'Sober requiere kernel Linux >=5.11.\')\n    cpuinfo = Path(\'/proc/cpuinfo\').read_text() if cpuinfo is None else cpuinfo\n    flags = [set(line.partition(\':\')[2].split()) for line in cpuinfo.splitlines()\n             if line.partition(\':\')[0].strip() == \'flags\']\n    if not flags or not all(\'sse4_1\' in item for item in flags):\n        raise RuntimeError(\'La CPU asignada no ofrece SSE4.1. No se puede añadir con un paquete; usar otra VM.\')\n    return {\'sse4_1\': True, \'sse4_2\': all(\'sse4_2\' in item for item in flags), \'kernel\': kernel}\n\n\ndef ensure_flatpak_system_bus(log):\n    """Flatpak consults parental-control policy even for --user installs."""\n    def ready():\n        result = subprocess.run([\'dbus-send\', \'--system\', \'--print-reply\',\n                                 \'--dest=org.freedesktop.DBus\', \'/org/freedesktop/DBus\',\n                                 \'org.freedesktop.DBus.ListNames\'], capture_output=True,\n                                text=True, timeout=5)\n        return result.returncode == 0\n    with (BASE / \'flatpak-system-bus.lock\').open(\'w\') as lock:\n        fcntl.flock(lock, fcntl.LOCK_EX)\n        if ready():\n            return\n        if Path(\'/run/dbus/system_bus_socket\').exists():\n            raise RuntimeError(\'El bus de sistema existente no responde; no se reemplaza automáticamente.\')\n        Path(\'/run/dbus\').mkdir(mode=0o755, parents=True, exist_ok=True)\n        command([\'dbus-daemon\', \'--system\', \'--fork\', \'--nopidfile\'], log, timeout=15)\n        if not ready():\n            raise RuntimeError(\'D-Bus de sistema no disponible para Flatpak; revisar install-roblox.log.\')\n\n\ndef install_roblox(session):\n    """Sober\'s official Flatpak only where the required kernel sandbox is available."""\n    requirements = sober_requirements()\n    result = subprocess.run([\'runuser\', \'-u\', session[\'user\'], \'--\', \'unshare\',\n                             \'--user\', \'--map-root-user\', \'--mount\', \'--pid\', \'--fork\', \'true\'],\n                            capture_output=True, text=True, timeout=10)\n    if result.returncode:\n        raise RuntimeError(\'Roblox Player necesita Sober/Flatpak, pero esta VM no permite \'\n                           \'el sandbox de namespaces. No se instala un cliente inoperante ni se \'\n                           \'desactiva el sandbox. Fuente: https://sober.vinegarhq.org/\')\n    log = BASE / \'install-roblox.log\'\n    command([\'apt-get\', \'update\', \'-qq\'], log)\n    command([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', \'flatpak\'], log)\n    try:\n        # unshare alone can succeed while this required proc mount is denied.\n        # Probe a real Bubblewrap sandbox before downloading large runtimes.\n        command([\'runuser\', \'-u\', session[\'user\'], \'--\', \'bwrap\',\n                 \'--unshare-user\', \'--unshare-pid\', \'--ro-bind\', \'/\', \'/\',\n                 \'--proc\', \'/proc\', \'--dev\', \'/dev\', \'--\', \'/usr/bin/true\'], log, timeout=15)\n    except RuntimeError as error:\n        raise RuntimeError(\'Flatpak está instalado, pero esta VM no permite el sandbox \'\n                           \'Bubblewrap con /proc. No se descargan los runtimes ni se fuerza \'\n                           \'Roblox sin sandbox. Revisar install-roblox.log; usar un entorno \'\n                           \'Linux que permita Flatpak.\') from error\n    ensure_flatpak_system_bus(log)\n    version = session.get(\'driver_version\', \'\')\n    if not re.fullmatch(r\'\\d+\\.\\d+(?:\\.\\d+)?\', version):\n        raise RuntimeError(\'No se conoce la versión NVIDIA exacta para Flatpak; no usar renderizado por software.\')\n    driver = \'nvidia-\' + version.replace(\'.\', \'-\')\n    # Flatpak cannot use the host\'s isolated LD_LIBRARY_PATH/ICD. It needs its\n    # own GL extension matching the loaded NVIDIA kernel driver exactly.\n    env = {k: v for k, v in session[\'env\'].items()\n           if k not in {\'LD_PRELOAD\', \'LD_LIBRARY_PATH\', \'VK_DRIVER_FILES\', \'VK_ICD_FILENAMES\', \'__EGL_VENDOR_LIBRARY_FILENAMES\', \'SELKIES_INTERPOSER\'}}\n    env[\'FLATPAK_GL_DRIVERS\'] = driver + \':default\'\n    prefix = [\'runuser\', \'-u\', session[\'user\'], \'--\', \'env\',\n              \'-u\', \'LD_PRELOAD\', \'-u\', \'LD_LIBRARY_PATH\', \'-u\', \'VK_DRIVER_FILES\', \'-u\', \'VK_ICD_FILENAMES\',\n              \'-u\', \'__EGL_VENDOR_LIBRARY_FILENAMES\',\n              *[f\'{k}={v}\' for k, v in env.items()], \'flatpak\', \'--user\']\n    command([*prefix, \'remote-add\', \'--if-not-exists\', \'flathub\',\n             \'https://flathub.org/repo/flathub.flatpakrepo\'], log)\n    command([*prefix, \'install\', \'-y\', \'--noninteractive\', \'flathub\',\n             f\'org.freedesktop.Platform.GL.{driver}//1.4\'], log, timeout=1200)\n    command([*prefix, \'install\', \'-y\', \'--noninteractive\', \'flathub\',\n             \'org.vinegarhq.Sober\'], log, timeout=1200)\n    # Runs only /usr/bin/true inside the real app sandbox, never opens Roblox.\n    command([*prefix, \'run\', \'--command=true\', \'org.vinegarhq.Sober\'], log, timeout=30)\n    wrapper = BASE / \'apps/sober-launch\'\n    wrapper.parent.mkdir(parents=True, exist_ok=True)\n    wrapper.write_text(\'#!/bin/sh\\nexec env -u LD_PRELOAD -u LD_LIBRARY_PATH \'\n                       \'-u VK_DRIVER_FILES -u VK_ICD_FILENAMES -u __EGL_VENDOR_LIBRARY_FILENAMES \'\n                       f\'FLATPAK_GL_DRIVERS={shlex.quote(driver + ":default")} \'\n                       \'flatpak run org.vinegarhq.Sober "$@"\\n\')\n    wrapper.chmod(0o755)\n    shortcut(session, \'Roblox - Sober\', str(wrapper))\n    (BASE / \'sober-requirements.json\').write_text(json.dumps({**requirements, \'flatpak_gl_driver\': driver,\n        \'sandbox_smoke_test\': True, \'game_validated\': False}, indent=2))\n    print(\'Sober instalado, sin abrir. Proyecto comunitario no respaldado por Roblox; \'\n          \'inicio de sesión, juego y controladores todavía requieren pruebas.\')\n\n\n_roblox_thread = None\n\n\ndef roblox_background(session):\n    """Keep desktop access available while downloading the Flatpak runtimes."""\n    global _roblox_thread\n    if _roblox_thread is not None and _roblox_thread.is_alive():\n        print(\'Sober: instalación ya en curso.\')\n        return _roblox_thread\n    def worker():\n        status = BASE / \'sober-status.txt\'\n        try:\n            with (BASE / \'sober-install.lock\').open(\'w\') as lock:\n                try:\n                    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n                except BlockingIOError:\n                    print(\'Sober: otra instalación está en curso.\', flush=True)\n                    return\n                install_roblox(session)\n                status.write_text(\'Instalado; abrir Roblox - Sober. Roblox se instala en el primer inicio.\\n\')\n        except Exception as error:\n            status.write_text(\'Instalación incompleta: \' + str(error) + \'\\n\')\n            print(\'Sober: \' + str(error), flush=True)\n    BASE.mkdir(exist_ok=True)\n    (BASE / \'sober-status.txt\').write_text(\'Instalando Flatpak, runtime NVIDIA y Sober en segundo plano.\\n\')\n    _roblox_thread = threading.Thread(target=worker, name=\'pcfree-sober\', daemon=True)\n    _roblox_thread.start()\n    print(\'Sober/Flatpak: descarga en segundo plano; el escritorio ya está disponible.\', flush=True)\n    return _roblox_thread\n\n\ndef install_app(option, session):\n    if os.geteuid() != 0:\n        raise RuntimeError(\'Ejecuta esta celda de instalación desde Colab.\')\n    if option not in OPTIONS and option not in APPIMAGES and option not in EXPERIMENTAL:\n        raise ValueError(\'Opción desconocida: \' + option)\n    BASE.mkdir(exist_ok=True)\n    log = BASE / f\'install-{option}.log\'\n    if option in EXPERIMENTAL:\n        {\'godot\': install_godot, \'vinegar\': install_vinegar, \'roblox\': install_roblox}[option](session)\n        return\n    if option in APPIMAGES:\n        if option == \'prism\':\n            install_app(\'java\', session)\n        official_appimage(option, session)\n        return\n    packages = OPTIONS[option]\n    if any(name.endswith(\':i386\') for name in packages):\n        command([\'dpkg\', \'--add-architecture\', \'i386\'], log)\n    command([\'apt-get\', \'update\', \'-qq\'], log)\n    command([\'apt-get\', \'install\', \'-y\', \'--no-install-recommends\', *packages], log)\n    if option == \'wine\':\n        shortcut(session, \'Wine Configuration\', \'/usr/bin/winecfg\', \'Settings\', \'wine\')\n        shortcut(session, \'Winetricks\', \'/usr/bin/winetricks\', \'Settings\', \'wine\')\n    elif option == \'lutris\':\n        shortcut(session, \'Lutris\', \'/usr/bin/lutris\', icon=\'lutris\')\n    elif option == \'deb-installer\':\n        shortcut(session, \'Instalador DEB\', \'/usr/bin/gdebi-gtk\', \'System\', \'package-x-generic\')\n    elif option == \'blender\':\n        shortcut(session, \'Blender\', \'/usr/bin/blender\', \'Graphics\', \'blender\')\n    print(f\'{option}: instalado, sin abrir aplicaciones. Registro: {log}\')\n    if option == \'wine\':\n        print(\'Wine no equivale a Windows. DXVK/VKD3D requieren Vulkan NVIDIA \'\n              \'verificado y bibliotecas NVIDIA de 32/64 bits; usa la celda de Steam \'\n              \'para preparar el soporte multiarch. No se instalan runners sin verificar.\')\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'option\', choices=sorted([*OPTIONS, *APPIMAGES, *EXPERIMENTAL]))\n    parser.add_argument(\'--session\', default=str(BASE / \'session.json\'))\n    args = parser.parse_args()\n    install_app(args.option, json.loads(Path(args.session).read_text()))\n', '10-export-worlds.py': '# Explicit, manual export. Never copies launcher accounts, browser profiles,\n# Steam, RustDesk, tokens, or the complete application home directory.\nimport zipfile, hashlib, json, datetime\nassert \'browser_env\' in globals(), \'Primero preparar el usuario de aplicaciones.\'\napp_home=Path(browser_env[\'HOME\']).resolve()\nworld_candidates=[]\nfor level in app_home.rglob(\'level.dat\'):\n    if level.is_symlink():\n        continue\n    world=level.parent.resolve()\n    if world.is_relative_to(app_home) and world.is_dir():\n        world_candidates.append(world)\nworld_candidates=sorted(set(world_candidates))\nprint(\'Mundos encontrados; seleccionar explícitamente los deseados:\')\nfor index,world in enumerate(world_candidates):\n    print(index,world)\n\n# Edit these values only after "Guardar y salir" in Minecraft.\nCONFIRM_GAME_SAVED_AND_CLOSED=False\nSELECTED_WORLD_INDICES=[]\nif not CONFIRM_GAME_SAVED_AND_CLOSED or not SELECTED_WORLD_INDICES:\n    print(\'Sin respaldo todavía. Guardar y salir del mundo, editar confirmación e índices, y repetir solo esta celda.\')\nelse:\n    assert all(isinstance(i,int) and 0<=i<len(world_candidates) for i in SELECTED_WORLD_INDICES)\n    stamp=datetime.datetime.now(datetime.timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n    destination=Path(\'/tmp\')/(\'PC-Free-worlds-\'+stamp+\'.zip\')\n    total=0\n    manifest={\'format\':\'pc-free-worlds-v1\',\'created_utc\':stamp,\'files\':[]}\n    with zipfile.ZipFile(destination,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n        for index in sorted(set(SELECTED_WORLD_INDICES)):\n            world=world_candidates[index]\n            for source in sorted(world.rglob(\'*\')):\n                if source.is_symlink() or not source.is_file() or source.name==\'session.lock\':\n                    continue\n                assert source.resolve().is_relative_to(world)\n                size=source.stat().st_size\n                total+=size\n                assert total<=2*1024**3, \'Respaldo mayor de 2 GiB; preparar exportación específica.\'\n                name=\'worlds/\'+str(index)+\'-\'+world.name+\'/\'+source.relative_to(world).as_posix()\n                digest=hashlib.file_digest(source.open(\'rb\'),\'sha256\').hexdigest()\n                archive.write(source,name)\n                manifest[\'files\'].append({\'path\':name,\'size\':size,\'sha256\':digest})\n        archive.writestr(\'manifest.json\',json.dumps(manifest,ensure_ascii=False,indent=2))\n    print(\'Respaldo creado:\',destination.name,\'| Bytes:\',destination.stat().st_size)\n    print(\'Solo mundos. Para mundos con mods, conservar también su lista/versiones de mods por separado.\')\n    from google.colab import files\n    files.download(str(destination))\n', '11-restore-worlds.py': "# Upload a previously exported ZIP without mounting Drive in the desktop.\nimport zipfile, json, hashlib, stat, datetime\nfrom google.colab import files\nfrom pathlib import PurePosixPath\nRESTORE_SAVES_DIRECTORY='' # Set the exact saves directory for this Minecraft instance.\nassert RESTORE_SAVES_DIRECTORY, 'Indicar primero la carpeta saves de la instancia de Minecraft.'\ntarget=Path(RESTORE_SAVES_DIRECTORY).expanduser().resolve()\napp_home=Path(browser_env['HOME']).resolve()\nassert target.is_relative_to(app_home), 'Restaurar solo dentro del home de aplicaciones.'\nassert target.name=='saves', 'La carpeta de destino debe llamarse saves.'\nuploaded=files.upload()\nassert len(uploaded)==1, 'Subir solamente un respaldo ZIP por intento.'\nfilename=next(iter(uploaded))\narchive_path=Path(filename)\nassert archive_path.suffix.lower()=='.zip'\nwith zipfile.ZipFile(archive_path) as archive:\n    infos=archive.infolist()\n    assert len({i.filename for i in infos})==len(infos), 'ZIP con rutas repetidas.'\n    assert sum(i.file_size for i in infos)<=2*1024**3, 'Respaldo demasiado grande.'\n    manifest=json.loads(archive.read('manifest.json'))\n    assert manifest['format']=='pc-free-worlds-v1'\n    expected={item['path']:item for item in manifest['files']}\n    assert set(expected)=={i.filename for i in infos if i.filename!='manifest.json' and not i.is_dir()}\n    for info in infos:\n        path=PurePosixPath(info.filename)\n        assert not path.is_absolute() and '..' not in path.parts and '\\\\' not in info.filename\n        assert not stat.S_ISLNK(info.external_attr>>16), 'No admitir enlaces simbólicos.'\n        if info.filename=='manifest.json' or info.is_dir():\n            continue\n        assert len(path.parts)>=3 and path.parts[0]=='worlds'\n        item=expected[info.filename]\n        assert info.file_size==item['size']\n        with archive.open(info) as source:\n            assert hashlib.file_digest(source,'sha256').hexdigest()==item['sha256'], 'Integridad incorrecta.'\n    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')\n    target.mkdir(parents=True,exist_ok=True)\n    restored={}\n    for item in manifest['files']:\n        path=PurePosixPath(item['path'])\n        name=path.parts[1]\n        if name not in restored:\n            directory=target/(name+'-restored-'+stamp)\n            directory.mkdir(exist_ok=False)\n            restored[name]=directory\n        output=restored[name].joinpath(*path.parts[2:])\n        assert output.resolve().is_relative_to(restored[name].resolve())\n        output.parent.mkdir(parents=True,exist_ok=True)\n        with archive.open(item['path']) as source, output.open('xb') as dest:\n            shutil.copyfileobj(source,dest)\n        account=pwd.getpwnam('pcfree-browser')\n        os.chown(output,account.pw_uid,account.pw_gid)\n    account=pwd.getpwnam('pcfree-browser')\n    for directory in restored.values():\n        for d in [directory,*[p for p in directory.rglob('*') if p.is_dir()]]:\n            os.chown(d,account.pw_uid,account.pw_gid)\n    print('Mundos restaurados en carpetas nuevas; no se sobrescribieron mundos existentes.')\n    print(*restored.values(),sep='\\n')\n"}
# Fuentes integradas: no clonar el repositorio ni ejecutar notebooks anteriores.
import os, pwd, shutil
from pathlib import Path
_core = {'__name__': 'pcfree_core'}
exec(compile(SOURCES['selkies-desktop.py'], 'selkies-desktop.py', 'exec'), _core)
def optional_module(name):
    namespace = {'__name__': 'pcfree_optional'}
    exec(compile(SOURCES[name], name, 'exec'), namespace)
    return namespace


In [ ]:
# @title Iniciar escritorio mínimo (no abre ventanas)
DOWNLOAD_FIREFOX = True # @param {type:"boolean"}
FRAMERATE = 60 # @param {type:"integer"}
VIDEO_BITRATE_KBPS = 5000 # @param {type:"integer"}
INSTALL_ROBLOX = True # @param {type:"boolean"}
SESSION = _core['bootstrap'](firefox=DOWNLOAD_FIREFOX, framerate=FRAMERATE, bitrate=VIDEO_BITRATE_KBPS)
browser_env = SESSION['env']  # Solo compatibilidad con respaldos manuales existentes.
_apps = optional_module('compat-apps.py')
_sober_task = _apps['roblox_background'](SESSION) if INSTALL_ROBLOX else None


## Opcionales — después de conectar
Cambia únicamente las opciones necesarias a `True` y ejecuta su celda.
Steam/Wine y juegos pueden necesitar dependencias de 32 bits y ajustes específicos.
El mapa de compatibilidad informa opciones no soportadas; no se fuerza Flatpak privilegiado.
Los mandos requieren `pcfree-run programa [argumentos]` y bibliotecas por arquitectura;
Steam ofrece fake-udev opcional. No se promete compatibilidad total de mandos o juegos.
`INSTALL_GAMEPADS` prepara mandos independientemente, sin descargar Steam.
Vinegar/Roblox Studio es experimental: instalación no equivale a ejecución validada.
Sober se incluye por defecto; `INSTALL_ROBLOX` en la celda de inicio permite desactivarlo.
Comprueba CPU x86-64, SSE4.1, kernel >=5.11 y el sandbox de namespaces de usuario.
Instala Flatpak y, cuando el sandbox lo permite, Sober y la extensión NVIDIA exacta.
No desactiva el sandbox ni cambia el controlador.
El sandbox Bubblewrap con `/proc` se prueba antes de descargar runtimes; en la VM
de desarrollo Colab rechazó ese montaje, por lo que Roblox no pudo ejecutarse.
SSE4.2 y Vulkan >=1.1 son recomendados; SSE4.1 no se instala por software.
Si la VM no cumple requisitos, conserva el escritorio y muestra un diagnóstico.
Roblox sigue siendo experimental: descarga el juego al abrir Sober por primera vez.
Usa también `pcfree-run /ruta/al/juego [argumentos]` para juegos descargados.
Chrome/Chromium reciente recomendado: mandos, fullscreen y pointer lock dependen del
navegador/dispositivo; Android y otros navegadores no ofrecen todas las funciones.
Para apuntar: activar **Gaming mode** y capturar el ratón dentro del escritorio.
Consultar la sección de controles del README antes de jugar desde Android.


In [ ]:
# @title Steam opcional — instala, no inicia el cliente
INSTALL_STEAM = False # @param {type:"boolean"}
STEAM_GAMEPADS = False # @param {type:"boolean"}
INSTALL_GAMEPADS = False # @param {type:"boolean"}
if (INSTALL_STEAM or INSTALL_GAMEPADS) and _sober_task is not None:
    _sober_task.join()  # Serializar APT con la descarga inicial de Sober.
if INSTALL_STEAM:
    SESSION['steam'] = {'gamepads': STEAM_GAMEPADS}
    optional_module('compat-steam.py')['install_steam'](SESSION)
if INSTALL_GAMEPADS and not (INSTALL_STEAM and STEAM_GAMEPADS):
    optional_module('compat-steam.py')['install_gamepads'](SESSION)


In [ ]:
# @title Aplicaciones opcionales — todo desactivado en Ejecutar todo
INSTALL_WINE = False # @param {type:"boolean"}
INSTALL_PRISM_MINECRAFT = False # @param {type:"boolean"}
INSTALL_LUTRIS = False # @param {type:"boolean"}
INSTALL_HEROIC = False # @param {type:"boolean"}
INSTALL_JAVA = False # @param {type:"boolean"}
INSTALL_GAMING_LIBS = False # @param {type:"boolean"}
INSTALL_DEB_INSTALLER = False # @param {type:"boolean"}
INSTALL_BLENDER = False # @param {type:"boolean"}
INSTALL_GODOT = False # @param {type:"boolean"}
INSTALL_VINEGAR = False # @param {type:"boolean"}
_options = {'wine': INSTALL_WINE, 'prism': INSTALL_PRISM_MINECRAFT,
            'lutris': INSTALL_LUTRIS, 'heroic': INSTALL_HEROIC,
            'java': INSTALL_JAVA, 'gaming-libs': INSTALL_GAMING_LIBS,
            'deb-installer': INSTALL_DEB_INSTALLER, 'blender': INSTALL_BLENDER,
            'godot': INSTALL_GODOT, 'vinegar': INSTALL_VINEGAR}
if any(_options.values()):
    if _sober_task is not None:
        _sober_task.join()
    for _option, _enabled in _options.items():
        if _enabled:
            _apps['install_app'](_option, SESSION)


## Estado y diagnóstico
Credenciales: salida privada de inicio; archivos locales separados en
`/tmp/pcfree-selkies/access.json` (0600). Contrato sin credenciales: `session.json`.
Registros por servicio en `/tmp/pcfree-selkies/`. Firefox informa éxito/error en la salida
y en `firefox-status.txt`; para reintentar una descarga fallida usa
`_core['firefox_background'](SESSION)`. Repetir el inicio no reinicia servicios activos;
Sober: consultar `sober-status.txt` e `install-roblox.log`; reintentar con
`_apps['roblox_background'](SESSION)` sin reiniciar el escritorio.
si hay un arranque parcial, inspecciona registros y usa una VM nueva.


## Respaldos manuales separados — desactivados
Solo mundos Minecraft, no cuentas/perfiles/contraseñas. Primero guardar y cerrar el juego.
Para exportar: activar la celda, inspeccionar índices y editar la selección dentro de ella.
Para restaurar: activar y definir una carpeta `saves` dentro del home de aplicaciones.
No se monta Drive. El cuaderno independiente **Drive-Backups.ipynb** conserva el flujo
manual de Drive, sin unirlo al arranque del escritorio.


In [ ]:
EXPORT_WORLDS = False
if EXPORT_WORLDS:
    # Explicit, manual export. Never copies launcher accounts, browser profiles,
    # Steam, RustDesk, tokens, or the complete application home directory.
    import zipfile, hashlib, json, datetime
    assert 'browser_env' in globals(), 'Primero preparar el usuario de aplicaciones.'
    app_home=Path(browser_env['HOME']).resolve()
    world_candidates=[]
    for level in app_home.rglob('level.dat'):
        if level.is_symlink():
            continue
        world=level.parent.resolve()
        if world.is_relative_to(app_home) and world.is_dir():
            world_candidates.append(world)
    world_candidates=sorted(set(world_candidates))
    print('Mundos encontrados; seleccionar explícitamente los deseados:')
    for index,world in enumerate(world_candidates):
        print(index,world)
    
    # Edit these values only after "Guardar y salir" in Minecraft.
    CONFIRM_GAME_SAVED_AND_CLOSED=False
    SELECTED_WORLD_INDICES=[]
    if not CONFIRM_GAME_SAVED_AND_CLOSED or not SELECTED_WORLD_INDICES:
        print('Sin respaldo todavía. Guardar y salir del mundo, editar confirmación e índices, y repetir solo esta celda.')
    else:
        assert all(isinstance(i,int) and 0<=i<len(world_candidates) for i in SELECTED_WORLD_INDICES)
        stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        destination=Path('/tmp')/('PC-Free-worlds-'+stamp+'.zip')
        total=0
        manifest={'format':'pc-free-worlds-v1','created_utc':stamp,'files':[]}
        with zipfile.ZipFile(destination,'w',compression=zipfile.ZIP_DEFLATED) as archive:
            for index in sorted(set(SELECTED_WORLD_INDICES)):
                world=world_candidates[index]
                for source in sorted(world.rglob('*')):
                    if source.is_symlink() or not source.is_file() or source.name=='session.lock':
                        continue
                    assert source.resolve().is_relative_to(world)
                    size=source.stat().st_size
                    total+=size
                    assert total<=2*1024**3, 'Respaldo mayor de 2 GiB; preparar exportación específica.'
                    name='worlds/'+str(index)+'-'+world.name+'/'+source.relative_to(world).as_posix()
                    digest=hashlib.file_digest(source.open('rb'),'sha256').hexdigest()
                    archive.write(source,name)
                    manifest['files'].append({'path':name,'size':size,'sha256':digest})
            archive.writestr('manifest.json',json.dumps(manifest,ensure_ascii=False,indent=2))
        print('Respaldo creado:',destination.name,'| Bytes:',destination.stat().st_size)
        print('Solo mundos. Para mundos con mods, conservar también su lista/versiones de mods por separado.')
        from google.colab import files
        files.download(str(destination))


In [ ]:
RESTORE_WORLDS = False
if RESTORE_WORLDS:
    # Upload a previously exported ZIP without mounting Drive in the desktop.
    import zipfile, json, hashlib, stat, datetime
    from google.colab import files
    from pathlib import PurePosixPath
    RESTORE_SAVES_DIRECTORY='' # Set the exact saves directory for this Minecraft instance.
    assert RESTORE_SAVES_DIRECTORY, 'Indicar primero la carpeta saves de la instancia de Minecraft.'
    target=Path(RESTORE_SAVES_DIRECTORY).expanduser().resolve()
    app_home=Path(browser_env['HOME']).resolve()
    assert target.is_relative_to(app_home), 'Restaurar solo dentro del home de aplicaciones.'
    assert target.name=='saves', 'La carpeta de destino debe llamarse saves.'
    uploaded=files.upload()
    assert len(uploaded)==1, 'Subir solamente un respaldo ZIP por intento.'
    filename=next(iter(uploaded))
    archive_path=Path(filename)
    assert archive_path.suffix.lower()=='.zip'
    with zipfile.ZipFile(archive_path) as archive:
        infos=archive.infolist()
        assert len({i.filename for i in infos})==len(infos), 'ZIP con rutas repetidas.'
        assert sum(i.file_size for i in infos)<=2*1024**3, 'Respaldo demasiado grande.'
        manifest=json.loads(archive.read('manifest.json'))
        assert manifest['format']=='pc-free-worlds-v1'
        expected={item['path']:item for item in manifest['files']}
        assert set(expected)=={i.filename for i in infos if i.filename!='manifest.json' and not i.is_dir()}
        for info in infos:
            path=PurePosixPath(info.filename)
            assert not path.is_absolute() and '..' not in path.parts and '\\' not in info.filename
            assert not stat.S_ISLNK(info.external_attr>>16), 'No admitir enlaces simbólicos.'
            if info.filename=='manifest.json' or info.is_dir():
                continue
            assert len(path.parts)>=3 and path.parts[0]=='worlds'
            item=expected[info.filename]
            assert info.file_size==item['size']
            with archive.open(info) as source:
                assert hashlib.file_digest(source,'sha256').hexdigest()==item['sha256'], 'Integridad incorrecta.'
        stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        target.mkdir(parents=True,exist_ok=True)
        restored={}
        for item in manifest['files']:
            path=PurePosixPath(item['path'])
            name=path.parts[1]
            if name not in restored:
                directory=target/(name+'-restored-'+stamp)
                directory.mkdir(exist_ok=False)
                restored[name]=directory
            output=restored[name].joinpath(*path.parts[2:])
            assert output.resolve().is_relative_to(restored[name].resolve())
            output.parent.mkdir(parents=True,exist_ok=True)
            with archive.open(item['path']) as source, output.open('xb') as dest:
                shutil.copyfileobj(source,dest)
            account=pwd.getpwnam('pcfree-browser')
            os.chown(output,account.pw_uid,account.pw_gid)
        account=pwd.getpwnam('pcfree-browser')
        for directory in restored.values():
            for d in [directory,*[p for p in directory.rglob('*') if p.is_dir()]]:
                os.chown(d,account.pw_uid,account.pw_gid)
        print('Mundos restaurados en carpetas nuevas; no se sobrescribieron mundos existentes.')
        print(*restored.values(),sep='\n')
